# Initialize

## Imports & paths

In [1]:
from pathlib import Path
import json
import numpy as np
import faiss
from rdflib import Graph
from rapidfuzz import fuzz

DATA_DIR = Path("/space_mounts/atai-hs25/dataset")
CODE_DIR = Path("/files/Final Project Submission")
METADATA_DIR = CODE_DIR / "metadata"
EMB_DIR = CODE_DIR / "embeddings"


## Load graph & metadata

In [2]:
print("--- Loading graph ---")
g = Graph()
g.parse(DATA_DIR / "graph.nt", format="nt")
print(f"Graph loaded: {len(g)} triples")

print("--- Loading labels ---")
with open(METADATA_DIR / "entity_labels.json", "r") as f:
    LABELS = json.load(f)

print("--- Loading IMDb map ---")
with open(METADATA_DIR / "imdb_map.json", "r") as f:
    IMDB_MAP = json.load(f)

print("--- Loading film entities ---")
FILM_URIS = set()
with open(METADATA_DIR / "film_entities.tsv", "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split("\t")
        uri = parts[0]
        FILM_URIS.add(uri)

print(f"Film entities: {len(FILM_URIS)}")


--- Loading graph ---
Graph loaded: 2413825 triples
--- Loading labels ---
--- Loading IMDb map ---
--- Loading film entities ---
Film entities: 175661


## Load embeddings & mappings

In [3]:
print("--- Loading embeddings ---")
E = np.load(EMB_DIR / "RFC_entity_embeds.npy")

ent_map = {}
idx2ent = {}

with open(METADATA_DIR / "entity_ids_ordered.tsv", "r") as f:
    for idx, line in enumerate(f):
        uri = line.strip()
        ent_map[uri] = idx
        idx2ent[idx] = uri

print(f"Entities in embedding space: {len(ent_map)}")

index = faiss.IndexFlatL2(E.shape[1])
index.add(E)
print("FAISS index ready.")


--- Loading embeddings ---
Entities in embedding space: 175661
FAISS index ready.


# Basic helpers: 
is_movie, label lookup, linking, embedding neighbors

In [4]:
def is_movie(uri: str) -> bool:
    return uri in FILM_URIS

def get_label(uri: str) -> str:
    lbl = LABELS.get(uri)
    if lbl:
        return lbl
    return uri.rsplit("/", 1)[-1]

def link_entity(name: str, must_be_movie: bool = False):
    """
    Very small linker:
    - exact label match (case-insensitive)
    - otherwise fuzzy over LABELS
    If must_be_movie=True, we only accept URIs in FILM_URIS.
    """
    name_lower = name.lower()

    # First: exact match
    for uri, lbl in LABELS.items():
        if lbl.lower() == name_lower:
            if not must_be_movie or is_movie(uri):
                return uri, lbl

    # Fuzzy match
    best_uri = None
    best_lbl = None
    best_score = 0

    for uri, lbl in LABELS.items():
        if must_be_movie and not is_movie(uri):
            continue
        score = fuzz.ratio(lbl.lower(), name_lower)
        if score > best_score:
            best_score = score
            best_uri = uri
            best_lbl = lbl

    if best_uri is None:
        return None, None

    # Only accept high-confidence matches
    if best_score < 80:
        return None, None

    return best_uri, best_lbl

def emb_neighbors(uri: str, k: int = 50):
    """
    One-hop embedding neighbors of a given URI.
    Returns list of (neighbor_uri, distance, neighbor_label, is_movie_flag).
    """
    if uri not in ent_map:
        return []

    idx = ent_map[uri]
    vec = E[idx].reshape(1, -1)
    D, I = index.search(vec, k)

    res = []
    for dist, idx2 in zip(D[0], I[0]):
        if idx2 == idx:
            continue
        n_uri = idx2ent[idx2]
        res.append((n_uri, float(dist), get_label(n_uri), is_movie(n_uri)))
    return res


Quick SPARQL helpers

In [5]:
RDFS_LABEL = "http://www.w3.org/2000/01/rdf-schema#label"

P57 = "http://www.wikidata.org/prop/direct/P57"   # director
P58 = "http://www.wikidata.org/prop/direct/P58"   # screenwriter
P136 = "http://www.wikidata.org/prop/direct/P136" # genre
P495 = "http://www.wikidata.org/prop/direct/P495" # country
P364 = "http://www.wikidata.org/prop/direct/P364" # language of work

def movies_with_same_property(seed_uri: str, prop_uri: str, limit: int = 100):
    """
    Very generic: given a seed movie and a property (director / country / etc),
    find other movies that share any value of that property.
    """
    q = f"""
    SELECT DISTINCT ?movie ?movieLabel WHERE {{
      <{seed_uri}> <{prop_uri}> ?v .
      ?movie <{prop_uri}> ?v .
      FILTER(?movie != <{seed_uri}>)
      ?movie <{RDFS_LABEL}> ?movieLabel .
      FILTER(LANG(?movieLabel) = "en")
    }} LIMIT {limit}
    """
    res = []
    for row in g.query(q):
        uri = str(row["movie"])
        if not is_movie(uri):
            continue
        res.append((uri, str(row["movieLabel"])))
    return res


Seed

In [6]:
TEST_SETS = {
    "musical_1": ["Singin' in the Rain", "Moulin Rouge"],
    "fellini": ["La Dolce Vita", "The Voice of the Moon"],
    "chicago_geisha_alice": ["Chicago", "Memoirs of a Geisha", "Alice in Wonderland"],
    "forrest_lotr": ["Forrest Gump", "The Lord of the Rings: The Fellowship of the Ring"],
    "japanese_kyoto": ["Twin Sisters of Kyoto"],
    "meryl_bio": ["Meryl Streep"],  # person, not forced to be movie
}

def link_test_set(name: str):
    titles = TEST_SETS[name]
    must_be_movie = name != "meryl_bio"

    uris = []
    print(f"=== Linking {name} ===")
    for t in titles:
        uri, lbl = link_entity(t, must_be_movie=must_be_movie)
        print(f"  {t!r} -> {uri} [{lbl}] (movie={is_movie(uri) if uri else None})")
        if uri:
            uris.append(uri)
    return uris

# Run once to see what URIs we are using
for key in TEST_SETS:
    _ = link_test_set(key)


=== Linking musical_1 ===
  "Singin' in the Rain" -> None [None] (movie=None)
  'Moulin Rouge' -> None [None] (movie=None)
=== Linking fellini ===
  'La Dolce Vita' -> None [None] (movie=None)
  'The Voice of the Moon' -> None [None] (movie=None)
=== Linking chicago_geisha_alice ===
  'Chicago' -> None [None] (movie=None)
  'Memoirs of a Geisha' -> None [None] (movie=None)
  'Alice in Wonderland' -> None [None] (movie=None)
=== Linking forrest_lotr ===
  'Forrest Gump' -> None [None] (movie=None)
  'The Lord of the Rings: The Fellowship of the Ring' -> None [None] (movie=None)
=== Linking japanese_kyoto ===
  'Twin Sisters of Kyoto' -> None [None] (movie=None)
=== Linking meryl_bio ===
  'Meryl Streep' -> http://www.wikidata.org/entity/Q873 [Meryl Streep] (movie=False)


# TEST

In [7]:
def debug_neighbors_for_set(name: str, k: int = 40):
    seed_uris = link_test_set(name)
    print()
    for seed in seed_uris:
        print(f"=== First-hop neighbors of {get_label(seed)} ({seed}) ===")
        neighbors = emb_neighbors(seed, k=k)
        movies = [n for n in neighbors if n[3]]
        non_movies = [n for n in neighbors if not n[3]]
        print(f"Total neighbors: {len(neighbors)}, movies: {len(movies)}, non-movies: {len(non_movies)}")
        print("Top 15 neighbors:")
        for (u, dist, lbl, is_mov) in neighbors[:15]:
            flag = "🎬" if is_mov else "✖"
            print(f"  {flag} {lbl:40s}  dist={dist:.4f}  {u}")

# For example:
debug_neighbors_for_set("musical_1", k=40)


=== Linking musical_1 ===
  "Singin' in the Rain" -> None [None] (movie=None)
  'Moulin Rouge' -> None [None] (movie=None)



# Recommend Function

In [8]:
def simple_embedding_rec_from_uris(seed_uris, k=80, topn=5):
    scores = {}
    for seed in seed_uris:
        if seed not in ent_map:
            continue
        idx = ent_map[seed]
        vec = E[idx].reshape(1, -1)
        D, I = index.search(vec, k)
        for dist, idx2 in zip(D[0], I[0]):
            if idx2 == idx:
                continue
            uri = idx2ent[idx2]
            if uri in seed_uris:
                continue
            if not is_movie(uri):
                continue
            sim = 1.0 / (1.0 + float(dist))
            scores[uri] = scores.get(uri, 0.0) + sim

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:topn]
    return [(uri, get_label(uri), score) for uri, score in ranked]

def simple_embedding_rec_from_titles(titles, k=80, topn=5, must_be_movie=True):
    seed_uris = []
    for t in titles:
        uri, _ = link_entity(t, must_be_movie=must_be_movie)
        if uri:
            seed_uris.append(uri)
    return simple_embedding_rec_from_uris(seed_uris, k=k, topn=topn)

# Example:
print("=== Simple embedding rec: musical_1 ===")
print(simple_embedding_rec_from_titles(TEST_SETS["musical_1"]))


=== Simple embedding rec: musical_1 ===
[]


# Try 01:

In [9]:
def normalize_uri(u: str) -> str:
    """
    Make URIs comparable across different files:
    - strip whitespace
    - remove leading/trailing angle brackets if present
    """
    if u is None:
        return None
    u = u.strip()
    if u.startswith("<") and u.endswith(">"):
        u = u[1:-1]
    return u


In [10]:
# --- Reload / normalize LABELS ---
with open(METADATA_DIR / "entity_labels.json", "r") as f:
    raw_labels = json.load(f)

LABELS = {}
for uri, lbl in raw_labels.items():
    norm_uri = normalize_uri(uri)
    LABELS[norm_uri] = lbl

print(f"Labels loaded: {len(LABELS)}")

# --- Reload / normalize FILM_URIS ---
FILM_URIS = set()
with open(METADATA_DIR / "film_entities.tsv", "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split("\t")
        uri_raw = parts[0]
        uri = normalize_uri(uri_raw)
        FILM_URIS.add(uri)

print(f"Film entities (normalized): {len(FILM_URIS)}")

# --- Reload / normalize embedding mappings ---
E = np.load(EMB_DIR / "RFC_entity_embeds.npy")

ent_map = {}
idx2ent = {}

with open(METADATA_DIR / "entity_ids_ordered.tsv", "r") as f:
    for idx, line in enumerate(f):
        uri_raw = line.strip()
        uri = normalize_uri(uri_raw)
        ent_map[uri] = idx
        idx2ent[idx] = uri

print(f"Entities in embedding space (normalized): {len(ent_map)}")

index = faiss.IndexFlatL2(E.shape[1])
index.add(E)
print("FAISS index ready with normalized URIs.")


Labels loaded: 129673
Film entities (normalized): 175661
Entities in embedding space (normalized): 175661
FAISS index ready with normalized URIs.


In [11]:
def is_movie(uri: str) -> bool:
    return uri in FILM_URIS

def get_label(uri: str) -> str:
    lbl = LABELS.get(uri)
    if lbl:
        return lbl
    return uri.rsplit("/", 1)[-1]

def link_entity(name: str, must_be_movie: bool = False):
    """
    Very small linker:
    - exact label match (case-insensitive)
    - otherwise fuzzy over LABELS
    If must_be_movie=True, we only accept URIs in FILM_URIS.
    """
    name_lower = name.lower()

    # 1) Exact match first
    for uri, lbl in LABELS.items():
        if lbl.lower() == name_lower:
            if not must_be_movie or is_movie(uri):
                return uri, lbl

    # 2) Fuzzy match
    best_uri = None
    best_lbl = None
    best_score = 0

    for uri, lbl in LABELS.items():
        if must_be_movie and not is_movie(uri):
            continue
        score = fuzz.ratio(lbl.lower(), name_lower)
        if score > best_score:
            best_score = score
            best_uri = uri
            best_lbl = lbl

    if best_uri is None:
        return None, None

    # Be slightly tolerant here, evaluation dataset titles sometimes differ a bit
    if best_score < 75:
        return None, None

    return best_uri, best_lbl

def emb_neighbors(uri: str, k: int = 50):
    """
    One-hop embedding neighbors of a given URI.
    Returns list of (neighbor_uri, distance, neighbor_label, is_movie_flag).
    """
    if uri not in ent_map:
        return []

    idx = ent_map[uri]
    vec = E[idx].reshape(1, -1)
    D, I = index.search(vec, k)

    res = []
    for dist, idx2 in zip(D[0], I[0]):
        if idx2 == idx:
            continue
        n_uri = idx2ent[idx2]
        res.append((n_uri, float(dist), get_label(n_uri), is_movie(n_uri)))
    return res


In [12]:
TEST_SETS = {
    "musical_1": ["Singin' in the Rain", "Moulin Rouge"],
    "fellini": ["La Dolce Vita", "The Voice of the Moon"],
    "chicago_geisha_alice": ["Chicago", "Memoirs of a Geisha", "Alice in Wonderland"],
    "forrest_lotr": ["Forrest Gump", "The Lord of the Rings: The Fellowship of the Ring"],
    "japanese_kyoto": ["Twin Sisters of Kyoto"],
    "meryl_bio": ["Meryl Streep"],
}

def link_test_set(name: str):
    titles = TEST_SETS[name]
    must_be_movie = name != "meryl_bio"

    uris = []
    print(f"=== Linking {name} ===")
    for t in titles:
        uri, lbl = link_entity(t, must_be_movie=must_be_movie)
        print(f"  {t!r} -> {uri} [{lbl}] (movie={is_movie(uri) if uri else None})")
        if uri:
            uris.append(uri)
    return uris

for key in TEST_SETS:
    _ = link_test_set(key)


=== Linking musical_1 ===
  "Singin' in the Rain" -> None [None] (movie=None)
  'Moulin Rouge' -> None [None] (movie=None)
=== Linking fellini ===
  'La Dolce Vita' -> None [None] (movie=None)
  'The Voice of the Moon' -> None [None] (movie=None)
=== Linking chicago_geisha_alice ===
  'Chicago' -> None [None] (movie=None)
  'Memoirs of a Geisha' -> None [None] (movie=None)
  'Alice in Wonderland' -> None [None] (movie=None)
=== Linking forrest_lotr ===
  'Forrest Gump' -> None [None] (movie=None)
  'The Lord of the Rings: The Fellowship of the Ring' -> None [None] (movie=None)
=== Linking japanese_kyoto ===
  'Twin Sisters of Kyoto' -> None [None] (movie=None)
=== Linking meryl_bio ===
  'Meryl Streep' -> http://www.wikidata.org/entity/Q873 [Meryl Streep] (movie=False)


In [13]:
print("=== Simple embedding rec: musical_1 ===")
print(simple_embedding_rec_from_titles(TEST_SETS["musical_1"]))


=== Simple embedding rec: musical_1 ===
[]


# debug helper

In [14]:
print(f"Number of film URIs: {len(FILM_URIS)}")

# show a few examples
from itertools import islice

print("Sample film URIs:")
for u in islice(FILM_URIS, 10):
    print("  ", repr(u))


Number of film URIs: 175661
Sample film URIs:
   '66155'
   '172645'
   '51917'
   '76760'
   '14545'
   '127198'
   '33955'
   '40075'
   '127539'
   '142464'


In [15]:
from heapq import nlargest

def debug_raw_link(name: str, topn: int = 20):
    name_lower = name.lower()
    candidates = []

    for uri, lbl in LABELS.items():
        score = fuzz.ratio(lbl.lower(), name_lower)
        candidates.append((score, uri, lbl, is_movie(uri)))

    best = nlargest(topn, candidates, key=lambda x: x[0])
    print(f"=== Raw fuzzy candidates for {name!r} ===")
    for score, uri, lbl, is_mov in best:
        print(f"  score={score:3d}  movie={is_mov}  {uri}  [{lbl}]")

debug_raw_link("Singin' in the Rain")
debug_raw_link("Moulin Rouge")
debug_raw_link("La Dolce Vita")


=== Raw fuzzy candidates for "Singin' in the Rain" ===


ValueError: Unknown format code 'd' for object of type 'float'

In [16]:
score=96  movie=False  http://www.wikidata.org/entity/Q34020  [La Dolce Vita]


SyntaxError: invalid syntax (1636676876.py, line 1)

In [17]:
u = "http://www.wikidata.org/entity/Q34020"
print("is_movie:", is_movie(u))
print("in FILM_URIS:", u in FILM_URIS)
print("raw present with angle brackets:", "<" + u + ">" in FILM_URIS)


is_movie: False
in FILM_URIS: False
raw present with angle brackets: False


In [18]:
is_movie: False
in FILM_URIS: False
raw present with angle brackets: True


SyntaxError: invalid syntax (3856630502.py, line 2)

# Try 02:
Print the first few lines of film_entities.tsv to verify the format.

Map the numeric IDs to URIs using idx2ent and reconstruct FILM_URIS.

Redefine is_movie.

Define the corrected debug_raw_link.

For the 6 test sets, respectively:

Try linking with must_be_movie=True;

Print the fuzzy top-5 candidates (without restricting to a specific movie) for titles that failed to link, so you can see "which one it most resembles".

## Check the link

In [19]:
from heapq import nlargest

# 1️⃣ Inspect raw film_entities.tsv
print("First 5 raw lines of film_entities.tsv:")
with open(METADATA_DIR / "film_entities.tsv", "r") as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print("  ", repr(line.strip()))

# 2️⃣ Rebuild FILM_URIS from numeric IDs -> URIs via idx2ent
FILM_URIS = set()
missing_idx = 0

with open(METADATA_DIR / "film_entities.tsv", "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        token = line.split("\t")[0]  # be robust to possible extra columns

        # Case 1: token is an integer index into the embedding/entity list
        if token.isdigit():
            idx = int(token)
            uri = idx2ent.get(idx)
            if uri is None:
                missing_idx += 1
                continue
        else:
            # Case 2: token might already be a URI or <URI>
            uri = normalize_uri(token)

        FILM_URIS.add(uri)

print(f"\nFilm URIs after remap: {len(FILM_URIS)}")
print(f"Missing indices (no uri in idx2ent): {missing_idx}")

# 3️⃣ Redefine is_movie using the new FILM_URIS
def is_movie(uri: str) -> bool:
    return uri in FILM_URIS

# 4️⃣ Better debug helper: fuzzy candidates without movie restriction
def debug_raw_link(name: str, topn: int = 10):
    name_lower = name.lower()
    candidates = []

    for uri, lbl in LABELS.items():
        score = fuzz.ratio(lbl.lower(), name_lower)
        candidates.append((score, uri, lbl, is_movie(uri)))

    best = nlargest(topn, candidates, key=lambda x: x[0])
    print(f"=== Raw fuzzy candidates for {name!r} ===")
    for score, uri, lbl, is_mov in best:
        print(f"  score={score:5.1f}  movie={is_mov}  {uri}  [{lbl}]")

# 5️⃣ Re-run linking tests with must_be_movie=True, and show fuzzy backups
print("\n========== LINKING TESTS WITH NEW FILM_URIS ==========\n")

for name, titles in TEST_SETS.items():
    must_be_movie = (name != "meryl_bio")
    print(f"=== Linking {name} (must_be_movie={must_be_movie}) ===")
    for t in titles:
        uri, lbl = link_entity(t, must_be_movie=must_be_movie)
        print(f"  {t!r} -> {uri} [{lbl}] (movie={is_movie(uri) if uri else None})")
        if uri is None:
            print("    ... top fuzzy candidates (ignoring must_be_movie):")
            debug_raw_link(t, topn=5)
    print()


First 5 raw lines of film_entities.tsv:
   'index\turi\tis_film\thas_plot'
   '0\thttp://www.wikidata.org/entity/Q0058036710383292\t1\t0'
   '1\thttp://www.wikidata.org/entity/Q0063555679015095\t1\t0'
   '2\thttp://www.wikidata.org/entity/Q0065575877878203\t1\t0'
   '3\thttp://www.wikidata.org/entity/Q0084631456723548\t1\t0'

Film URIs after remap: 175661
Missing indices (no uri in idx2ent): 0

========== LINKING TESTS WITH NEW FILM_URIS ==========

=== Linking musical_1 (must_be_movie=True) ===
  "Singin' in the Rain" -> None [None] (movie=None)
    ... top fuzzy candidates (ignoring must_be_movie):
=== Raw fuzzy candidates for "Singin' in the Rain" ===
  score=100.0  movie=False  http://www.wikidata.org/entity/Q131736016  [Singin' in the Rain]
  score=100.0  movie=False  http://www.wikidata.org/entity/Q7523531  [Singin' in the Rain]
  score=100.0  movie=False  http://www.wikidata.org/entity/Q309153  [Singin' in the Rain]
  score= 89.5  movie=False  http://www.wikidata.org/entity/Q170

## Try to fix the link

In [20]:
from heapq import nlargest

# 1️⃣ Rebuild FILM_URIS using the URI + is_film columns
FILM_URIS = set()

film_path = METADATA_DIR / "film_entities.tsv"
with open(film_path, "r") as f:
    header = next(f).strip().split("\t")
    print("film_entities header:", header)
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) < 3:
            continue
        index, uri, is_film, *rest = parts
        # only keep rows explicitly marked as films
        if is_film == "1":
            FILM_URIS.add(normalize_uri(uri))

print("\nFilm URIs (from uri + is_film):", len(FILM_URIS))

# 2️⃣ Sanity-check: are some well-known movie URIs in FILM_URIS?
test_movie_uris = [
    # Singin' in the Rain candidates
    "http://www.wikidata.org/entity/Q131736016",
    "http://www.wikidata.org/entity/Q7523531",
    "http://www.wikidata.org/entity/Q309153",
    # Moulin Rouge candidates
    "http://www.wikidata.org/entity/Q151030",
    "http://www.wikidata.org/entity/Q1508611",
    "http://www.wikidata.org/entity/Q193573",
    # La Dolce Vita / Voice of the Moon
    "http://www.wikidata.org/entity/Q18407",
    "http://www.wikidata.org/entity/Q18442",
    # Chicago / Memoirs of a Geisha / Alice in Wonderland
    "http://www.wikidata.org/entity/Q189889",
    "http://www.wikidata.org/entity/Q656285",
    "http://www.wikidata.org/entity/Q45839",
    "http://www.wikidata.org/entity/Q189875",
    # Forrest Gump / LOTR
    "http://www.wikidata.org/entity/Q134773",
    "http://www.wikidata.org/entity/Q3077690",
    "http://www.wikidata.org/entity/Q127367",
    # Twin Sisters of Kyoto
    "http://www.wikidata.org/entity/Q1458080",
]

print("\nSanity-check: known movie URIs in FILM_URIS?")
for u in test_movie_uris:
    print(" ", u, "->", u in FILM_URIS)

# 3️⃣ Redefine is_movie using the new FILM_URIS
def is_movie(uri: str) -> bool:
    return uri in FILM_URIS

# 4️⃣ Improved debug helper: top fuzzy candidates regardless of movie flag
def debug_raw_link(name: str, topn: int = 10):
    name_lower = name.lower()
    candidates = []

    for uri, lbl in LABELS.items():
        score = fuzz.ratio(lbl.lower(), name_lower)
        candidates.append((score, uri, lbl, is_movie(uri)))

    best = nlargest(topn, candidates, key=lambda x: x[0])
    print(f"=== Raw fuzzy candidates for {name!r} ===")
    for score, uri, lbl, is_mov in best:
        print(f"  score={score:5.1f}  movie={is_mov}  {uri}  [{lbl}]")

# 5️⃣ Re-run linking tests with must_be_movie=True/False
print("\n========== LINKING TESTS WITH NEW FILM_URIS (URI column) ==========\n")

for name, titles in TEST_SETS.items():
    must_be_movie = (name != "meryl_bio")
    print(f"=== Linking {name} (must_be_movie={must_be_movie}) ===")
    for t in titles:
        uri, lbl = link_entity(t, must_be_movie=must_be_movie)
        print(f"  {t!r} -> {uri} [{lbl}] (movie={is_movie(uri) if uri else None})")
        if uri is None:
            print("    ... top fuzzy candidates (ignoring must_be_movie):")
            debug_raw_link(t, topn=5)
    print()


film_entities header: ['index', 'uri', 'is_film', 'has_plot']

Film URIs (from uri + is_film): 12245

Sanity-check: known movie URIs in FILM_URIS?
  http://www.wikidata.org/entity/Q131736016 -> False
  http://www.wikidata.org/entity/Q7523531 -> False
  http://www.wikidata.org/entity/Q309153 -> True
  http://www.wikidata.org/entity/Q151030 -> False
  http://www.wikidata.org/entity/Q1508611 -> True
  http://www.wikidata.org/entity/Q193573 -> True
  http://www.wikidata.org/entity/Q18407 -> True
  http://www.wikidata.org/entity/Q18442 -> True
  http://www.wikidata.org/entity/Q189889 -> True
  http://www.wikidata.org/entity/Q656285 -> False
  http://www.wikidata.org/entity/Q45839 -> True
  http://www.wikidata.org/entity/Q189875 -> False
  http://www.wikidata.org/entity/Q134773 -> True
  http://www.wikidata.org/entity/Q3077690 -> False
  http://www.wikidata.org/entity/Q127367 -> True
  http://www.wikidata.org/entity/Q1458080 -> True

========== LINKING TESTS WITH NEW FILM_URIS (URI column) =

## Check embedding coverage + simple recommendation
Ensure that embeddings, ent_map, idx2ent, and FAISS index are all available;

Check if the seed movie URIs mentioned above are in ent_map;

Define a simple "multi-seed embedding recommendation" function, performing an average vector + KNN;

Run this function once for 6 seed sets, printing the top 10 movie recommendations (already filtered using is_movie).

In [21]:
import numpy as np
import faiss
from collections import defaultdict

# 1️⃣ Ensure embeddings and FAISS index are loaded
# If you already loaded them earlier in the notebook, this will just reuse them.

if "E" not in globals():
    # Adjust paths if needed
    E = np.load(CODE_DIR / "embeddings/RFC_entity_embeds.npy")

if "ent_map" not in globals() or "idx2ent" not in globals():
    ent_map = {}
    idx2ent = {}
    with open(DATA_DIR / "embeddings/entity_ids.del") as f:
        for line in f:
            i, u = line.strip().split("\t")
            i = int(i)
            ent_map[u] = i
            idx2ent[i] = u

if "faiss_index" not in globals():
    faiss_index = faiss.IndexFlatL2(E.shape[1])
    faiss_index.add(E)

print("Embedding matrix shape:", E.shape)
print("Number of ent_map entries:", len(ent_map))

# 2️⃣ Check if our seed movie URIs have embeddings
seed_uri_groups = {
    "musical_1": [
        "http://www.wikidata.org/entity/Q309153",   # Singin' in the Rain
        "http://www.wikidata.org/entity/Q1508611",  # Moulin Rouge
    ],
    "fellini": [
        "http://www.wikidata.org/entity/Q18407",    # La Dolce Vita
        "http://www.wikidata.org/entity/Q18442",    # The Voice of the Moon
    ],
    "chicago_geisha_alice": [
        "http://www.wikidata.org/entity/Q189889",   # Chicago
        "http://www.wikidata.org/entity/Q45839",    # Memoirs of a Geisha
        "http://www.wikidata.org/entity/Q1761330",  # Alice in Wonderland
    ],
    "forrest_lotr": [
        "http://www.wikidata.org/entity/Q134773",   # Forrest Gump
        "http://www.wikidata.org/entity/Q127367",   # LOTR 1
    ],
    "japanese_kyoto": [
        "http://www.wikidata.org/entity/Q1458080",  # Twin Sisters of Kyoto
    ],
}

print("\n=== Embedding coverage for seed movie URIs ===")
for name, uris in seed_uri_groups.items():
    print(f"\n{name}:")
    for u in uris:
        has_emb = u in ent_map
        print("  ", u, "-> embedding:", has_emb)

# 3️⃣ Helper to go from title -> movie URI -> embedding index
def title_to_movie_idx(title: str):
    """
    Link a user-given title to a movie URI, then return its embedding index.
    """
    uri, lbl = link_entity(title, must_be_movie=True)
    if uri is None:
        return None, None, None
    idx = ent_map.get(uri)
    return uri, lbl, idx

# 4️⃣ Simple multi-seed embedding recommendation
def embedding_recommendation(seed_titles, k_neighbors=200, top_n=10):
    """
    1. Link each seed title to a movie URI.
    2. Average the embeddings of all seeds that have embeddings.
    3. Run FAISS KNN.
    4. Filter to movie URIs, and exclude the seeds themselves.
    """
    print(f"\n=== Simple embedding recommendation for: {seed_titles} ===")

    seed_uris = []
    seed_idxs = []

    # Step 1: link and collect embeddings
    for t in seed_titles:
        uri, lbl, idx = title_to_movie_idx(t)
        print(f"  Seed title {t!r} -> uri={uri}  label={lbl}  idx={idx}")
        if uri is not None and idx is not None:
            seed_uris.append(uri)
            seed_idxs.append(idx)

    if not seed_idxs:
        print("  ❌ No seeds have embeddings, cannot recommend.")
        return []

    # Step 2: average seed vectors
    seed_vecs = E[np.array(seed_idxs)]
    mean_vec = seed_vecs.mean(axis=0).astype("float32").reshape(1, -1)

    # Step 3: FAISS nearest neighbors
    D, I = faiss_index.search(mean_vec, k_neighbors)

    # Step 4: filter to movies & not in seeds
    results = []
    seen = set()

    for idx in I[0]:
        uri = idx2ent[idx]
        if uri in seed_uris:
            continue
        if not is_movie(uri):
            continue
        if uri in seen:
            continue
        seen.add(uri)

        lbl = LABELS.get(uri, uri.split("/")[-1])
        results.append((uri, lbl))

        if len(results) >= top_n:
            break

    print("  Top recommendations:")
    for uri, lbl in results:
        print("    ->", lbl, "(", uri, ")")

    return results

# 5️⃣ Run the simple embedding rec on our test groups
TEST_REC_SETS = {
    "musical_1": ["Singin' in the Rain", "Moulin Rouge"],
    "fellini": ["La Dolce Vita", "The Voice of the Moon"],
    "chicago_geisha_alice": ["Chicago", "Memoirs of a Geisha", "Alice in Wonderland"],
    "forrest_lotr": ["Forrest Gump", "The Lord of the Rings: The Fellowship of the Ring"],
    "japanese_kyoto": ["Twin Sisters of Kyoto"],
}

all_results = {}

for name, titles in TEST_REC_SETS.items():
    print("\n" + "=" * 60)
    print(f"Running embedding recommendations for: {name}")
    recs = embedding_recommendation(titles, k_neighbors=300, top_n=10)
    all_results[name] = recs


Embedding matrix shape: (175660, 128)
Number of ent_map entries: 175661

=== Embedding coverage for seed movie URIs ===

musical_1:
   http://www.wikidata.org/entity/Q309153 -> embedding: False
   http://www.wikidata.org/entity/Q1508611 -> embedding: False

fellini:
   http://www.wikidata.org/entity/Q18407 -> embedding: False
   http://www.wikidata.org/entity/Q18442 -> embedding: False

chicago_geisha_alice:
   http://www.wikidata.org/entity/Q189889 -> embedding: False
   http://www.wikidata.org/entity/Q45839 -> embedding: False
   http://www.wikidata.org/entity/Q1761330 -> embedding: False

forrest_lotr:
   http://www.wikidata.org/entity/Q134773 -> embedding: False
   http://www.wikidata.org/entity/Q127367 -> embedding: False

japanese_kyoto:
   http://www.wikidata.org/entity/Q1458080 -> embedding: False

Running embedding recommendations for: musical_1

=== Simple embedding recommendation for: ["Singin' in the Rain", 'Moulin Rouge'] ===
  Seed title "Singin' in the Rain" -> uri=http:

In [22]:
from rapidfuzz import process, fuzz

# 1️⃣ Build a label list restricted to entities that HAVE embeddings
#    (so any URI we pick from here will have an index in ent_map and a row in E)
EMB_LABELS = {}
for uri, lbl in LABELS.items():
    if uri in ent_map:
        EMB_LABELS[uri] = lbl

EMB_URI_LIST = list(EMB_LABELS.keys())
EMB_LABEL_LIST = [EMB_LABELS[u] for u in EMB_URI_LIST]

print(f"Total entities with embeddings and labels: {len(EMB_LABELS)}")

# 2️⃣ Helper: show fuzzy candidates (embedding-aware)
def debug_fuzzy_with_emb(title, limit=10):
    print(f"\n=== Embedding-aware candidates for {title!r} ===")
    matches = process.extract(
        title,
        EMB_LABEL_LIST,
        scorer=fuzz.WRatio,
        limit=limit
    )
    for label, score, idx in matches:
        uri = EMB_URI_LIST[idx]
        mov = is_movie(uri)
        print(f"  score={score:5.1f}  movie={mov!s:5}  {uri}  [{label}]")

# 3️⃣ Main linker for recommendation seeds
def link_for_embedding(title, prefer_movie=True, top_k=20):
    """
    Link a user-given title to an entity that:
      - definitely has an embedding (uri in ent_map),
      - preferably is a film (is_movie(uri) == True).
    """
    matches = process.extract(
        title,
        EMB_LABEL_LIST,
        scorer=fuzz.WRatio,
        limit=top_k
    )
    if not matches:
        return None

    candidates = []
    for label, score, idx in matches:
        uri = EMB_URI_LIST[idx]
        mov = is_movie(uri)
        candidates.append({
            "uri": uri,
            "label": label,
            "score": score,
            "is_movie": mov,
        })

    # Prefer movie candidates with embeddings (all have embeddings by construction)
    if prefer_movie:
        movie_cands = [c for c in candidates if c["is_movie"]]
        if movie_cands:
            best = max(movie_cands, key=lambda c: c["score"])
            return best

    # Fallback: any entity with embedding
    best = max(candidates, key=lambda c: c["score"])
    return best

# 4️⃣ New embedding recommendation that uses link_for_embedding
def embedding_recommendation_v2(seed_titles, k_neighbors=300, top_n=10):
    """
    1. For each seed title, use link_for_embedding to get an entity with embedding.
    2. Average their embeddings.
    3. Run FAISS KNN.
    4. Keep only URIs that are films and not in the seed set.
    """
    print(f"\n=== Embedding recommendation v2 for: {seed_titles} ===")

    seed_uris = []
    seed_idxs = []

    # Step 1: resolve seeds
    for t in seed_titles:
        best = link_for_embedding(t, prefer_movie=True, top_k=50)
        if best is None:
            print(f"  Seed title {t!r} -> ❌ no embedding-aware match")
            continue
        uri = best["uri"]
        label = best["label"]
        idx = ent_map[uri]
        print(f"  Seed title {t!r} -> uri={uri}  label={label}  idx={idx}  movie={best['is_movie']}")
        seed_uris.append(uri)
        seed_idxs.append(idx)

    if not seed_idxs:
        print("  ❌ No seeds have embeddings, cannot recommend.")
        return []

    # Step 2: average vectors
    seed_vecs = E[np.array(seed_idxs)]
    mean_vec = seed_vecs.mean(axis=0).astype("float32").reshape(1, -1)

    # Step 3: FAISS search
    D, I = faiss_index.search(mean_vec, k_neighbors)

    # Step 4: filter to films & not in seeds
    results = []
    seen = set()
    for idx in I[0]:
        uri = idx2ent[idx]
        if uri in seed_uris:
            continue
        if not is_movie(uri):
            continue
        if uri in seen:
            continue
        seen.add(uri)
        lbl = LABELS.get(uri, uri.split("/")[-1])
        results.append((uri, lbl))
        if len(results) >= top_n:
            break

    print("  Top recommendations:")
    for uri, lbl in results:
        print("    ->", lbl, "(", uri, ")")

    return results

# 5️⃣ Try it on the evaluation seed sets
TEST_REC_SETS = {
    "musical_1": ["Singin' in the Rain", "Moulin Rouge"],
    "fellini": ["La Dolce Vita", "The Voice of the Moon"],
    "chicago_geisha_alice": ["Chicago", "Memoirs of a Geisha", "Alice in Wonderland"],
    "forrest_lotr": ["Forrest Gump", "The Lord of the Rings: The Fellowship of the Ring"],
    "japanese_kyoto": ["Twin Sisters of Kyoto"],
}

all_results_v2 = {}
for name, titles in TEST_REC_SETS.items():
    print("\n" + "="*70)
    print(f"Running embedding_recommendation_v2 for: {name}")
    # Optional: inspect fuzzy candidates for the first title
    debug_fuzzy_with_emb(titles[0])
    recs = embedding_recommendation_v2(titles, k_neighbors=400, top_n=10)
    all_results_v2[name] = recs


Total entities with embeddings and labels: 0

Running embedding_recommendation_v2 for: musical_1

=== Embedding-aware candidates for "Singin' in the Rain" ===

=== Embedding recommendation v2 for: ["Singin' in the Rain", 'Moulin Rouge'] ===
  Seed title "Singin' in the Rain" -> ❌ no embedding-aware match
  Seed title 'Moulin Rouge' -> ❌ no embedding-aware match
  ❌ No seeds have embeddings, cannot recommend.

Running embedding_recommendation_v2 for: fellini

=== Embedding-aware candidates for 'La Dolce Vita' ===

=== Embedding recommendation v2 for: ['La Dolce Vita', 'The Voice of the Moon'] ===
  Seed title 'La Dolce Vita' -> ❌ no embedding-aware match
  Seed title 'The Voice of the Moon' -> ❌ no embedding-aware match
  ❌ No seeds have embeddings, cannot recommend.

Running embedding_recommendation_v2 for: chicago_geisha_alice

=== Embedding-aware candidates for 'Chicago' ===

=== Embedding recommendation v2 for: ['Chicago', 'Memoirs of a Geisha', 'Alice in Wonderland'] ===
  Seed tit

In [24]:
import random
import re

print("===== BASIC STATS =====")
print(f"#LABELS keys: {len(LABELS)}")
print(f"#idx2ent entries (embedding entities): {len(idx2ent)}")

# 1️⃣ Sample some LABELS keys and idx2ent values
def sample_dict(d, n=10):
    items = list(d.items())
    if len(items) > n:
        items = random.sample(items, n)
    return items

print("\n===== SAMPLE LABELS KEYS =====")
for uri, lbl in sample_dict(LABELS, 10):
    print("LABEL URI:", uri, " | label:", lbl)

print("\n===== SAMPLE idx2ent VALUES (embedding entities) =====")
for idx, uri in sample_dict(idx2ent, 10):
    print("EMB idx:", idx, " | uri:", uri)

# 2️⃣ Normalization function: strip <>, spaces, extract QID if possible
def norm_uri(u, mode="full"):
    if u is None:
        return None
    u = u.strip()
    # strip angle brackets if present
    if u.startswith("<") and u.endswith(">"):
        u = u[1:-1].strip()
    if mode == "full":
        return u
    elif mode == "qid":
        m = re.search(r"(Q\d+)", u)
        return m.group(1) if m else None
    else:
        return u

# 3️⃣ Check domains to see what kind of URIs we have
def domain_of(u):
    u = norm_uri(u, mode="full")
    if u is None:
        return None
    if "wikidata.org" in u:
        return "wikidata"
    if "ddis.ch" in u:
        return "ddis"
    if u.startswith("Q") and "://" not in u:
        return "bare_q"
    if "://" not in u:
        return "no_scheme"
    return "other"

label_domains = {}
for uri in list(LABELS.keys())[:5000]:  # sample first 5000 for speed
    dom = domain_of(uri)
    label_domains[dom] = label_domains.get(dom, 0) + 1

idx_domains = {}
for idx, uri in list(idx2ent.items())[:5000]:
    dom = domain_of(uri)
    idx_domains[dom] = idx_domains.get(dom, 0) + 1

print("\n===== DOMAIN STATS (first 5000) =====")
print("LABEL domains:", label_domains)
print("EMB idx2ent domains:", idx_domains)

# 4️⃣ Build FULL-URI based intersection (current failing case)
norm_labels_full = {norm_uri(u, "full") for u in LABELS.keys()}
norm_idx_full = {norm_uri(u, "full") for u in idx2ent.values()}
inter_full = norm_labels_full & norm_idx_full

print("\n===== FULL URI INTERSECTION =====")
print(f"#normalized LABEL URIs: {len(norm_labels_full)}")
print(f"#normalized EMB URIs:   {len(norm_idx_full)}")
print(f"#FULL-URI intersection: {len(inter_full)}")

if inter_full:
    print("Example FULL-URI intersection elements (up to 10):")
    for u in list(inter_full)[:10]:
        print("  ", u)

# 5️⃣ Try QID-based intersection
label_qids = {norm_uri(u, "qid") for u in LABELS.keys()}
idx_qids = {norm_uri(u, "qid") for u in idx2ent.values()}

# Remove None
label_qids.discard(None)
idx_qids.discard(None)

inter_qid = label_qids & idx_qids

print("\n===== QID INTERSECTION =====")
print(f"#LABEL QIDs: {len(label_qids)}")
print(f"#EMB   QIDs: {len(idx_qids)}")
print(f"#QID   intersection: {len(inter_qid)}")

if inter_qid:
    print("Example QID intersection elements (up to 10):")
    for q in list(inter_qid)[:10]:
        print("  ", q)

# 6️⃣ If we have QID intersection, show how URIs look on both sides
if inter_qid:
    some_q = list(inter_qid)[0]
    print(f"\n===== EXAMPLE QID {some_q} ON BOTH SIDES =====")
    # Find any LABEL URI with this QID
    lbl_uri_example = next(u for u in LABELS.keys() if norm_uri(u, "qid") == some_q)
    emb_uri_example = next(u for u in idx2ent.values() if norm_uri(u, "qid") == some_q)
    print("LABEL URI example:", lbl_uri_example)
    print("EMB   URI example:", emb_uri_example)


===== BASIC STATS =====
#LABELS keys: 129673
#idx2ent entries (embedding entities): 175661

===== SAMPLE LABELS KEYS =====
LABEL URI: http://www.wikidata.org/entity/Q5298630  | label: Dorothy Short
LABEL URI: http://www.wikidata.org/entity/Q421471  | label: Aksel Hennie
LABEL URI: http://www.wikidata.org/entity/Q5913311  | label: Ignacio Huang
LABEL URI: http://www.wikidata.org/entity/Q1751525  | label: Lynn Reynolds
LABEL URI: http://www.wikidata.org/entity/Q36107  | label: Muhammad Ali
LABEL URI: http://www.wikidata.org/entity/Q6247211  | label: John Mattson
LABEL URI: http://www.wikidata.org/entity/Q51963292  | label: Marvel Cinematic Universe Phase One
LABEL URI: http://www.wikidata.org/entity/Q744002  | label: Jeff Chase
LABEL URI: http://www.wikidata.org/entity/Q275728  | label: Lies Welsin
LABEL URI: http://www.wikidata.org/entity/Q1085177  | label: City Girl

===== SAMPLE idx2ent VALUES (embedding entities) =====
EMB idx: 99447  | uri: 99446	http://www.wikidata.org/entity/Q3370

In [25]:
import re
import numpy as np
import faiss
from collections import defaultdict

# 1️⃣ Small helper: normalize and extract QID
def extract_qid(u: str) -> str | None:
    """Extract QID (e.g., 'Q18407') from any URI / raw string."""
    if u is None:
        return None
    u = u.strip()
    # Drop angle brackets if any
    if u.startswith("<") and u.endswith(">"):
        u = u[1:-1].strip()
    m = re.search(r"(Q\d+)", u)
    return m.group(1) if m else None

# 2️⃣ Build QID -> embedding index mapping from idx2ent
qid2idx: dict[str, int] = {}
for idx, raw in idx2ent.items():
    qid = extract_qid(raw)
    if qid is not None and qid not in qid2idx:
        qid2idx[qid] = idx

print(f"Built qid2idx for {len(qid2idx)} entities with embeddings.")

# 3️⃣ Build QID -> label mapping from LABELS
qid2label: dict[str, str] = {}
for uri, lbl in LABELS.items():
    qid = extract_qid(uri)
    if qid is not None and qid not in qid2label:
        qid2label[qid] = lbl

print(f"Built qid2label for {len(qid2label)} labeled entities.")

# 4️⃣ Build film QID set from FILM_URIS
film_qids: set[str] = set()
for uri in FILM_URIS:
    qid = extract_qid(uri)
    if qid is not None:
        film_qids.add(qid)

print(f"Film QIDs: {len(film_qids)}")

# 5️⃣ Make sure embeddings are float32 for FAISS
E32 = E.astype("float32", copy=False)

# Build FAISS index (if not already built)
index = faiss.IndexFlatL2(E32.shape[1])
index.add(E32)
print(f"FAISS index built with {index.ntotal} vectors.")


def embedding_recommendation_qid(seed_titles, k_neighbors=200, top_k=10, print_debug=True):
    """
    Recommend movies similar to the given seed titles using:
    - linker to get seed URIs
    - QID to connect to embeddings and labels
    - pooling neighbors in embedding space
    """
    if print_debug:
        print("\n=== QID-based embedding recommendation for:", seed_titles, "===")

    # 1) Resolve seed titles -> movie URIs via linker
    seed_qids = []
    seed_idxs = []
    seen_qids = set()

    for title in seed_titles:
        uri, lbl, is_movie = link_movie(title, must_be_movie=True)
        if print_debug:
            print(f"  Seed title {title!r} -> uri={uri}  label={lbl}  movie={is_movie}")

        if not uri or not is_movie:
            continue

        qid = extract_qid(uri)
        if not qid:
            continue

        if qid in qid2idx:
            idx = qid2idx[qid]
            seed_qids.append(qid)
            seed_idxs.append(idx)
            seen_qids.add(qid)
        else:
            if print_debug:
                print(f"    ⚠ QID {qid} has no embedding index.")

    if not seed_idxs:
        if print_debug:
            print("  ❌ No seeds have embeddings, cannot recommend.")
        return []

    # 2) For each seed, gather neighbors in embedding space
    candidate_scores = defaultdict(float)

    for qid, idx in zip(seed_qids, seed_idxs):
        vec = E32[idx : idx + 1]  # shape (1, d)
        D, I = index.search(vec, k_neighbors)

        if print_debug:
            print(f"\n  Neighbors for seed QID {qid} (top 5 shown):")

        for rank, (dist, j) in enumerate(zip(D[0], I[0])):
            if j == idx:
                continue  # skip self
            raw = idx2ent[j]
            q_neighbor = extract_qid(raw)
            if not q_neighbor:
                continue

            # Only keep films
            if q_neighbor not in film_qids:
                continue

            # Exclude original seeds
            if q_neighbor in seen_qids:
                continue

            # Convert distance to similarity (simple heuristic)
            score = -float(dist)
            candidate_scores[q_neighbor] += score

            if print_debug and rank < 5:
                lbl = qid2label.get(q_neighbor, q_neighbor)
                print(f"    rank={rank:3d}  qid={q_neighbor}  score={score:.4f}  label={lbl}")

    if not candidate_scores:
        if print_debug:
            print("  ⚠ No film neighbors found; returning empty list.")
        return []

    # 3) Rank candidates by aggregated score
    ranked = sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    results = []
    if print_debug:
        print("\n  Final ranked recommendations:")

    for qid, score in ranked:
        lbl = qid2label.get(qid, qid)
        # reconstruct a canonical wikidata URI for output
        uri = f"http://www.wikidata.org/entity/{qid}"
        if print_debug:
            print(f"    -> {lbl}  (qid={qid}, score={score:.4f})")
        results.append({"label": lbl, "uri": uri, "qid": qid, "score": score})

    return results


Built qid2idx for 175660 entities with embeddings.
Built qid2label for 129174 labeled entities.
Film QIDs: 12245
FAISS index built with 175660 vectors.


In [27]:
from rapidfuzz import fuzz
import re
from collections import defaultdict
import numpy as np
import faiss

# ----------------------------------------------------
# 1) Helper: extract QID
# ----------------------------------------------------
def extract_qid(u: str) -> str | None:
    """Extract QID (e.g. 'Q18407') from any URI / raw string."""
    if u is None:
        return None
    u = u.strip()
    if u.startswith("<") and u.endswith(">"):
        u = u[1:-1].strip()
    m = re.search(r"(Q\d+)", u)
    return m.group(1) if m else None


# ----------------------------------------------------
# 2) Movie-aware linker (LABELS + FILM_URIS)
# ----------------------------------------------------
def link_movie(name: str, must_be_movie: bool = True, score_cutoff: float = 80.0):
    """
    Very simple movie linker:
    - exact label match first
    - then fuzzy over LABELS
    - optionally restrict to URIs that are known films (FILM_URIS)
    Returns: (uri, label, is_movie) or (None, None, None)
    """
    if not name:
        return None, None, None

    name_norm = name.strip().lower()
    best = None  # (score, uri, label, is_movie)

    # 1) Exact label match
    for uri, lbl in LABELS.items():
        if lbl.lower() == name_norm:
            is_movie = uri in FILM_URIS
            if must_be_movie and not is_movie:
                # label exact but not a film, skip if we require a movie
                continue
            return uri, lbl, is_movie

    # 2) Fuzzy label match
    for uri, lbl in LABELS.items():
        lbl_norm = lbl.lower()
        score = fuzz.ratio(lbl_norm, name_norm)
        if score < score_cutoff:
            continue

        is_movie = uri in FILM_URIS
        if must_be_movie and not is_movie:
            continue

        if best is None or score > best[0]:
            best = (score, uri, lbl, is_movie)

    if best is None:
        return None, None, None

    _, uri, lbl, is_movie = best
    return uri, lbl, is_movie


# ----------------------------------------------------
# 3) QID-based embedding recommendation
# ----------------------------------------------------
def embedding_recommendation_qid(seed_titles, k_neighbors=200, top_k=10, print_debug=True):
    """
    Recommend movies similar to the given seed titles using:
      - link_movie() to get seed URIs
      - QID to connect to embeddings and labels
      - pooling neighbors in embedding space
    Assumes:
      - qid2idx, qid2label, film_qids, E32, index, idx2ent already exist.
    """
    if print_debug:
        print("\n=== QID-based embedding recommendation for:", seed_titles, "===")

    seed_qids = []
    seed_idxs = []
    seen_qids = set()

    # 1) Resolve seeds via linker
    for title in seed_titles:
        uri, lbl, is_movie = link_movie(title, must_be_movie=True)
        if print_debug:
            print(f"  Seed title {title!r} -> uri={uri}  label={lbl}  movie={is_movie}")

        if not uri or not is_movie:
            continue

        qid = extract_qid(uri)
        if not qid:
            continue

        if qid in qid2idx:
            idx = qid2idx[qid]
            seed_qids.append(qid)
            seed_idxs.append(idx)
            seen_qids.add(qid)
        else:
            if print_debug:
                print(f"    ⚠ QID {qid} has no embedding index.")

    if not seed_idxs:
        if print_debug:
            print("  ❌ No seeds have embeddings, cannot recommend.")
        return []

    # 2) Gather neighbors for each seed in embedding space
    candidate_scores = defaultdict(float)

    for qid, idx in zip(seed_qids, seed_idxs):
        vec = E32[idx : idx + 1]  # shape (1, d)
        D, I = index.search(vec, k_neighbors)

        if print_debug:
            print(f"\n  Neighbors for seed QID {qid} (top 5 shown):")

        for rank, (dist, j) in enumerate(zip(D[0], I[0])):
            if j == idx:
                continue  # skip self
            raw = idx2ent[j]
            q_neighbor = extract_qid(raw)
            if not q_neighbor:
                continue

            # only keep films
            if q_neighbor not in film_qids:
                continue

            # exclude original seeds
            if q_neighbor in seen_qids:
                continue

            # convert distance to a similarity-like score
            score = -float(dist)
            candidate_scores[q_neighbor] += score

            if print_debug and rank < 5:
                lbl = qid2label.get(q_neighbor, q_neighbor)
                print(f"    rank={rank:3d}  qid={q_neighbor}  score={score:.4f}  label={lbl}")

    if not candidate_scores:
        if print_debug:
            print("  ⚠ No film neighbors found; returning empty list.")
        return []

    # 3) Rank candidates by aggregated score
    ranked = sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    results = []
    if print_debug:
        print("\n  Final ranked recommendations:")

    for qid, score in ranked:
        lbl = qid2label.get(qid, qid)
        uri = f"http://www.wikidata.org/entity/{qid}"
        if print_debug:
            print(f"    -> {lbl}  (qid={qid}, score={score:.4f})")
        results.append({"label": lbl, "uri": uri, "qid": qid, "score": score})

    return results


# ----------------------------------------------------
# 4) Quick test on the 5 official seed sets
# ----------------------------------------------------
test_sets = {
    "musical_1": ["Singin' in the Rain", "Moulin Rouge"],
    "fellini": ["La Dolce Vita", "The Voice of the Moon"],
    "chicago_geisha_alice": ["Chicago", "Memoirs of a Geisha", "Alice in Wonderland"],
    "forrest_lotr": ["Forrest Gump", "The Lord of the Rings: The Fellowship of the Ring"],
    "japanese_kyoto": ["Twin Sisters of Kyoto"],
}

for name, seeds in test_sets.items():
    print("\n" + "=" * 70)
    print("Running embedding_recommendation_qid for:", name)
    recs = embedding_recommendation_qid(seeds, k_neighbors=200, top_k=5, print_debug=True)
    print("Top results:", [r["label"] for r in recs])



Running embedding_recommendation_qid for: musical_1

=== QID-based embedding recommendation for: ["Singin' in the Rain", 'Moulin Rouge'] ===
  Seed title "Singin' in the Rain" -> uri=http://www.wikidata.org/entity/Q309153  label=Singin' in the Rain  movie=True
  Seed title 'Moulin Rouge' -> uri=http://www.wikidata.org/entity/Q1508611  label=Moulin Rouge  movie=True

  Neighbors for seed QID Q309153 (top 5 shown):

  Neighbors for seed QID Q1508611 (top 5 shown):

  Final ranked recommendations:
    -> Lontano da dove  (qid=Q15617274, score=-0.0178)
    -> Youngblood Reloaded  (qid=Q6916451754674039, score=-0.0195)
    -> Paranormal Activity 2  (qid=Q912082, score=-0.0221)
    -> Real Steel  (qid=Q261759, score=-0.0223)
    -> The Hobbit: The Battle of the Five Armies  (qid=Q919649, score=-0.0232)
Top results: ['Lontano da dove', 'Youngblood Reloaded', 'Paranormal Activity 2', 'Real Steel', 'The Hobbit: The Battle of the Five Armies']

Running embedding_recommendation_qid for: fellini


# Try 03:

In [29]:
# AND-first, OR-fallback embedding recommendation over film QIDs
# Assumes the following already exist in the notebook:
# - link_movie(title, must_be_movie=True/False) -> (uri, label, is_movie)
# - film_qids: set of QIDs that are films
# - qid2idx: dict QID -> embedding row index
# - qid2label: dict QID -> label string
# - E32: float32 embedding matrix (shape [n_entities, d])
# - index: FAISS IndexFlatL2 built on E32

import numpy as np

# Build reverse mapping once: idx -> QID
idx2qid = {idx: qid for qid, idx in qid2idx.items()}

def embedding_recommendation_and_or_qid(seed_titles, k_neighbors=200, top_k=5, print_debug=True):
    """
    AND-first, OR-fallback recommendation:
    1. Link each seed title to a movie QID with an embedding index.
    2. For each seed, get k_neighbors nearest film neighbors in embedding space.
    3. AND step: candidates appearing in all seeds' neighbor sets.
       - Score = average distance across seeds (lower is better).
    4. If AND set is empty, fallback to OR:
       - Candidates in union of neighbor sets.
       - Score = min distance to any seed (closest seed wins).
    5. Return top_k candidates (as list of dicts).
    """
    # 1️⃣ Link seeds to movie QIDs + embedding indices
    seed_info = []  # list of (title, qid, idx, label)
    if print_debug:
        print(f"=== AND/OR embedding recommendation for: {seed_titles} ===")
    for title in seed_titles:
        uri, lbl, is_movie = link_movie(title, must_be_movie=True)
        if not uri or not is_movie:
            if print_debug:
                print(f"  Seed {title!r} -> ❌ not linked to a movie (uri={uri}, movie={is_movie})")
            continue
        qid = uri.rsplit('/', 1)[-1]
        idx = qid2idx.get(qid)
        if idx is None:
            if print_debug:
                print(f"  Seed {title!r} -> uri={uri} qid={qid} but ❌ no embedding index")
            continue
        seed_info.append((title, qid, idx, lbl))
        if print_debug:
            print(f"  Seed {title!r} -> uri={uri} qid={qid} label={lbl} idx={idx}")

    if not seed_info:
        if print_debug:
            print("  ❌ No valid seed movies with embeddings.")
        return []

    # 2️⃣ For each seed, collect nearest film neighbors
    per_seed_neighbors = {}  # seed_qid -> {cand_qid: distance}
    seed_qids = [s[1] for s in seed_info]

    for title, seed_qid, idx, lbl in seed_info:
        vec = E32[idx].reshape(1, -1)
        D, I = index.search(vec, k_neighbors)
        cand_dict = {}

        # Collect neighbors that are films and not the seed itself
        for dist, emb_idx in zip(D[0], I[0]):
            if emb_idx < 0:
                continue
            cand_qid = idx2qid.get(int(emb_idx))
            if cand_qid is None:
                continue
            if cand_qid == seed_qid:
                continue
            if cand_qid not in film_qids:
                continue
            # Only keep best distance per candidate for this seed
            if cand_qid not in cand_dict or dist < cand_dict[cand_qid]:
                cand_dict[cand_qid] = float(dist)

        per_seed_neighbors[seed_qid] = cand_dict

        if print_debug:
            print(f"\n  Neighbors for seed {title!r} (first 5):")
            # sort by distance ascending and show first 5
            preview = sorted(cand_dict.items(), key=lambda kv: kv[1])[:5]
            for cand_qid, dist in preview:
                cand_label = qid2label.get(cand_qid, cand_qid)
                print(f"    dist={dist:.4f}  qid={cand_qid}  label={cand_label}")

    # 3️⃣ AND step: intersection of candidate sets
    neighbor_sets = [set(d.keys()) for d in per_seed_neighbors.values() if d]
    if neighbor_sets:
        intersection = set.intersection(*neighbor_sets)
    else:
        intersection = set()

    if print_debug:
        print(f"\n  AND-candidates intersection size: {len(intersection)}")

    candidates_scores = []

    if intersection:
        # AND semantics: must appear in all neighbor sets
        for cand_qid in intersection:
            # distances from all seeds that have this candidate
            dists = []
            for seed_qid in seed_qids:
                dist = per_seed_neighbors.get(seed_qid, {}).get(cand_qid, None)
                if dist is not None:
                    dists.append(dist)
            if not dists:
                continue
            # Aggregate: average distance (lower is better)
            agg_dist = float(np.mean(dists))
            candidates_scores.append((cand_qid, agg_dist))
        if print_debug:
            print("  Using AND semantics (common neighbors of all seeds).")
    else:
        # 4️⃣ OR fallback: union of neighbor sets
        union = set()
        for s in neighbor_sets:
            union |= s
        if print_debug:
            print(f"  AND set empty, using OR-union with {len(union)} candidates.")

        for cand_qid in union:
            # OR semantics: use min distance to any seed (closest seed wins)
            dists = []
            for seed_qid in seed_qids:
                dist = per_seed_neighbors.get(seed_qid, {}).get(cand_qid, None)
                if dist is not None:
                    dists.append(dist)
            if not dists:
                continue
            agg_dist = float(min(dists))
            candidates_scores.append((cand_qid, agg_dist))

    # 5️⃣ Sort by distance ascending and take top_k
    candidates_scores.sort(key=lambda x: x[1])
    top = candidates_scores[:top_k]

    results = []
    if print_debug:
        print("\n  Final ranked recommendations:")
    for cand_qid, dist in top:
        label = qid2label.get(cand_qid, cand_qid)
        if print_debug:
            print(f"    -> {label}  (qid={cand_qid}, agg_dist={dist:.4f})")
        results.append({"qid": cand_qid, "label": label, "distance": dist})

    return results


# ======================
# Quick tests for the 5 eval-style queries (film seeds only for now)
# ======================

TEST_QUERIES = {
    "musical_1": [
        "Singin' in the Rain",
        "Moulin Rouge",
    ],
    "fellini": [
        "La Dolce Vita",
        "The Voice of the Moon",
    ],
    "chicago_geisha_alice": [
        "Chicago",
        "Memoirs of a Geisha",
        "Alice in Wonderland",
    ],
    "forrest_lotr": [
        "Forrest Gump",
        "The Lord of the Rings: The Fellowship of the Ring",
    ],
    "japanese_kyoto": [
        "Twin Sisters of Kyoto",
    ],
    # NOTE: Meryl Streep is an actor, not a film; we will handle actor→films later.
    # For now we skip her in this AND/OR film-based test or treat separately.
}

for name, seeds in TEST_QUERIES.items():
    print("\n" + "=" * 70)
    print(f"Running embedding_recommendation_and_or_qid for: {name}")
    recs = embedding_recommendation_and_or_qid(seeds, k_neighbors=200, top_k=5, print_debug=True)
    print("Top labels:", [r["label"] for r in recs])



Running embedding_recommendation_and_or_qid for: musical_1
=== AND/OR embedding recommendation for: ["Singin' in the Rain", 'Moulin Rouge'] ===
  Seed "Singin' in the Rain" -> uri=http://www.wikidata.org/entity/Q309153 qid=Q309153 label=Singin' in the Rain idx=91742
  Seed 'Moulin Rouge' -> uri=http://www.wikidata.org/entity/Q1508611 qid=Q1508611 label=Moulin Rouge idx=34227

  Neighbors for seed "Singin' in the Rain" (first 5):
    dist=0.0178  qid=Q15617274  label=Lontano da dove
    dist=0.0195  qid=Q6916451754674039  label=Youngblood Reloaded
    dist=0.0221  qid=Q912082  label=Paranormal Activity 2
    dist=0.0223  qid=Q261759  label=Real Steel
    dist=0.0232  qid=Q919649  label=The Hobbit: The Battle of the Five Armies

  Neighbors for seed 'Moulin Rouge' (first 5):
    dist=0.2065  qid=Q2362866  label=Sleepaway Camp III: Teenage Wasteland
    dist=0.2280  qid=Q1645003  label=Kaho Naa... Pyaar Hai
    dist=0.2941  qid=Q1570822  label=The Subject Was Roses
    dist=0.3007  qid=Q

In [31]:
# Graph-based debugging cell for recommendations

import os
from agent.graph_executor import GraphExecutor
from agent.composer import Composer
from agent.constants import PREFIXES

# If you have a Config that knows the data root, you can use it,
# but DO NOT pass the Config object into GraphExecutor directly.
try:
    from agent.config import Config
    cfg = Config()
except ImportError:
    cfg = None

# 1) Decide the actual RDF graph path.
#    >>> IMPORTANT: adjust this to your real file <<<
#
# Typical patterns in this project are something like:
#   /space_mounts/atai-hs25/dataset/graph.nt
#   /space_mounts/atai-hs25/dataset/movies_kg.nt
#   /space_mounts/atai-hs25/dataset/movies_kg.ttl
#
# Use cfg.data_root if available to construct the path.
if cfg is not None and hasattr(cfg, "data_root"):
    data_root = cfg.data_root
else:
    # Fallback: hard-code or use the default root you know
    data_root = "/space_mounts/atai-hs25/dataset"

# TODO: change this file name to the one you actually have
GRAPH_PATH = os.path.join(data_root, "graph.nt")

print("Using RDF graph file:", GRAPH_PATH)

# 2) Instantiate graph executor with a *path string*, not Config
graph = GraphExecutor(GRAPH_PATH)
composer = Composer()

def run_sparql(query: str, limit: int = 10):
    """
    Run a SPARQL SELECT via GraphExecutor and return a list of dicts.
    Also pretty-print the first few rows for quick inspection.
    """
    print("---- SPARQL QUERY ----")
    print(query.strip())
    print("----------------------")

    res = graph.execute_query(query)
    rows = []
    for row in res:
        rows.append({str(k): str(v) for k, v in row.items()})

    print(f"Total rows: {len(rows)}")
    for i, row in enumerate(rows[:limit]):
        print(f"Row {i+1}: {row}")
    if len(rows) > limit:
        print(f"... {len(rows) - limit} more rows not shown")
    return rows

def debug_graph_neighbors(seed_uri: str, limit: int = 10):
    """
    For a given seed movie URI, compose the graph-based rec query
    and inspect what neighbors come back from the local RDF graph.
    """
    print(f"\n=== Graph neighbors for seed: {seed_uri} ===")
    q = composer.compose_graph_rec_query(seed_uri)
    rows = run_sparql(q, limit=limit)
    return rows

# Example usage once you know the seed URIs (from your linker):
#   debug_graph_neighbors("http://www.wikidata.org/entity/Q309153")   # Singin' in the Rain
#   debug_graph_neighbors("http://www.wikidata.org/entity/Q1508611")  # Moulin Rouge
#   debug_graph_neighbors("http://www.wikidata.org/entity/Q18407")    # La Dolce Vita
#   debug_graph_neighbors("http://www.wikidata.org/entity/Q18442")    # The Voice of the Moon


Using RDF graph file: /space_mounts/atai-hs25/dataset/graph.nt


In [33]:
# Example usage once you know the seed URIs (from your linker):
debug_graph_neighbors("http://www.wikidata.org/entity/Q309153")   # Singin' in the Rain
debug_graph_neighbors("http://www.wikidata.org/entity/Q1508611")  # Moulin Rouge
debug_graph_neighbors("http://www.wikidata.org/entity/Q18407")    # La Dolce Vita
debug_graph_neighbors("http://www.wikidata.org/entity/Q18442")    # The Voice of the Moon


=== Graph neighbors for seed: http://www.wikidata.org/entity/Q309153 ===
---- SPARQL QUERY ----
PREFIX ddis: <http://ddis.ch/atai/>
    PREFIX wd: <http://www.wikidata.org/entity/>
    PREFIX wdt: <http://www.wikidata.org/prop/direct/>
    PREFIX schema: <http://schema.org/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX p: <http://www.wikidata.org/prop/>
    PREFIX ps: <http://www.wikidata.org/prop/statement/>
    PREFIX psv: <http://www.wikidata.org/prop/statement/value/>
 SELECT DISTINCT ?movie ?rating WHERE {
                { <http://www.wikidata.org/entity/Q309153> wdt:P136 ?g . ?movie wdt:P136 ?g . } UNION
                { <http://www.wikidata.org/entity/Q309153> wdt:P57 ?d . ?movie wdt:P57 ?d . }
                ?movie wdt:P31 wd:Q11424 . OPTIONAL { ?movie ddis:rating ?rating . }
            } LIMIT 50
----------------------
Total rows: 50
Row 1: {'movie': 'http://www.wikidata.org/entity/Q1062362'}
Row 2: {'movie': 'http://www.wikidata.org/entity/Q685245'

[{'movie': 'http://www.wikidata.org/entity/Q23925083'},
 {'movie': 'http://www.wikidata.org/entity/Q123701240'},
 {'movie': 'http://www.wikidata.org/entity/Q513716'},
 {'movie': 'http://www.wikidata.org/entity/Q3227962'},
 {'movie': 'http://www.wikidata.org/entity/Q6815682'},
 {'movie': 'http://www.wikidata.org/entity/Q478076', 'rating': '7.1'},
 {'movie': 'http://www.wikidata.org/entity/Q1091097'},
 {'movie': 'http://www.wikidata.org/entity/Q23793836'},
 {'movie': 'http://www.wikidata.org/entity/Q602256'},
 {'movie': 'http://www.wikidata.org/entity/Q468877'},
 {'movie': 'http://www.wikidata.org/entity/Q2363955', 'rating': '6.3'},
 {'movie': 'http://www.wikidata.org/entity/Q1612502'},
 {'movie': 'http://www.wikidata.org/entity/Q746612'},
 {'movie': 'http://www.wikidata.org/entity/Q25139'},
 {'movie': 'http://www.wikidata.org/entity/Q1162433'},
 {'movie': 'http://www.wikidata.org/entity/Q28891'},
 {'movie': 'http://www.wikidata.org/entity/Q368577'},
 {'movie': 'http://www.wikidata.org/e

In [34]:
from collections import defaultdict

# We assume these already exist from previous cells:
# - LABELS: dict[uri -> label]
# - link_movie(title, must_be_movie=True)
# - graph: GraphExecutor
# - composer: Composer

# 1) Build a simple QID -> label map from LABELS
qid2label = {uri.split("/")[-1]: lbl for uri, lbl in LABELS.items()}

def graph_neighbors_for_qid(qid: str, limit: int = 200, print_debug: bool = False):
    """
    Given a movie QID, query the RDF graph for neighbors:
    - Same genre (wdt:P136) OR same director (wdt:P57)
    - Must be film (wdt:P31 wd:Q11424)
    Return: dict[qid -> list of ratings from different matches]
    """
    uri = f"http://www.wikidata.org/entity/{qid}"
    query = composer.compose_graph_rec_query(uri)
    rows = graph.execute_query(query)

    if print_debug:
        print(f"\n=== Graph neighbors for seed QID {qid} ({uri}) ===")
        print(f"Total raw rows: {len(rows)}")

    neighbors_ratings = defaultdict(list)

    for row in rows:
        m_uri = str(row["movie"])
        m_qid = m_uri.split("/")[-1]
        if m_qid == qid:
            # skip the seed itself
            continue

        rating_val = None
        if "rating" in row and row["rating"] is not None:
            try:
                rating_val = float(row["rating"])
            except Exception:
                rating_val = None

        neighbors_ratings[m_qid].append(rating_val)

    # truncate if needed (ordering is arbitrary here; scoring happens later)
    if limit is not None and len(neighbors_ratings) > limit:
        # simple truncation – we will sort later again anyway
        trimmed_keys = list(neighbors_ratings.keys())[:limit]
        neighbors_ratings = {k: neighbors_ratings[k] for k in trimmed_keys}

    if print_debug:
        sample = list(neighbors_ratings.items())[:5]
        print("Sample neighbors (up to 5):")
        for m_qid, ratings in sample:
            lbl = qid2label.get(m_qid, f"[no label for {m_qid}]")
            print(f"  {m_qid}  label={lbl!r}  ratings={ratings}")

    return neighbors_ratings


def graph_recommendation_and_or(
    seed_titles,
    top_k: int = 5,
    use_and_when_possible: bool = True,
    print_debug: bool = True,
):
    """
    Graph-based recommendation using:
      - link_movie() to get QIDs for seeds
      - composer + GraphExecutor to find neighbors per seed
      - AND semantics: intersection of neighbor sets across all seeds (if non-empty)
      - otherwise OR semantics: union of neighbors

    Ranking:
      - first by how many seeds a candidate is connected to (coverage, desc)
      - then by average rating (desc; unrated treated as 0)
      - finally by label (for stable ordering)
    """
    if print_debug:
        print("\n" + "=" * 70)
        print("Graph AND/OR recommendation for seeds:", seed_titles)

    # 1) Link seeds to URIs/QIDs
    seed_qids = []
    for title in seed_titles:
        uri, lbl, is_movie = link_movie(title, must_be_movie=True)
        if uri is None:
            if print_debug:
                print(f"  Seed {title!r} could not be linked to a movie; skipping.")
            continue
        qid = uri.split("/")[-1]
        seed_qids.append(qid)
        if print_debug:
            print(f"  Seed {title!r} -> uri={uri} qid={qid} label={lbl} movie={is_movie}")

    if not seed_qids:
        if print_debug:
            print("  ❌ No seeds could be linked to movies. Returning empty list.")
        return []

    # 2) Collect graph neighbors for each seed QID
    per_seed_neighbors = {}
    for qid in seed_qids:
        neigh = graph_neighbors_for_qid(qid, limit=200, print_debug=print_debug)
        per_seed_neighbors[qid] = neigh

    # 3) Build candidate sets
    seed_sets = [set(d.keys()) for d in per_seed_neighbors.values()]

    if use_and_when_possible and len(seed_sets) > 1:
        # intersection of all neighbor sets
        common = set.intersection(*seed_sets) if seed_sets else set()
        if print_debug:
            sizes = [len(s) for s in seed_sets]
            print(f"\n  Per-seed neighbor sizes: {sizes}")
            print(f"  AND intersection size: {len(common)}")

        if common:
            candidate_qids = common
            if print_debug:
                print("  Using AND semantics (common neighbors across all seeds).")
        else:
            # fallback to OR-union if intersection is empty
            candidate_qids = set.union(*seed_sets)
            if print_debug:
                print("  AND set empty, using OR-union.")
    else:
        # only one seed or AND disabled -> OR
        candidate_qids = set.union(*seed_sets)
        if print_debug:
            print("  Using OR semantics (single seed or AND disabled).")

    # 4) Aggregate coverage + rating for each candidate
    #    coverage = how many seeds connect to this candidate
    #    avg_rating = mean of available ratings across seeds (unrated -> None)
    stats = {}
    for cand_qid in candidate_qids:
        coverage = 0
        all_ratings = []
        for seed_qid, neigh_dict in per_seed_neighbors.items():
            if cand_qid in neigh_dict:
                coverage += 1
                # there can be multiple ratings entries per seed
                for r in neigh_dict[cand_qid]:
                    if r is not None:
                        all_ratings.append(r)

        avg_rating = sum(all_ratings) / len(all_ratings) if all_ratings else 0.0
        stats[cand_qid] = {
            "coverage": coverage,
            "avg_rating": avg_rating,
        }

    # 5) Rank candidates
    def rank_key(qid):
        st = stats[qid]
        lbl = qid2label.get(qid, "")
        # negative because we want highest coverage & rating first
        return (-st["coverage"], -st["avg_rating"], lbl)

    ranked_qids = sorted(candidate_qids, key=rank_key)

    # 6) Build result objects, skip seeds themselves
    seed_qid_set = set(seed_qids)
    results = []
    for qid in ranked_qids:
        if qid in seed_qid_set:
            continue
        uri = f"http://www.wikidata.org/entity/{qid}"
        lbl = qid2label.get(qid, uri)
        st = stats[qid]
        results.append(
            {
                "qid": qid,
                "uri": uri,
                "label": lbl,
                "coverage": st["coverage"],
                "avg_rating": st["avg_rating"],
            }
        )
        if len(results) >= top_k:
            break

    if print_debug:
        print("\n  Final ranked recommendations:")
        for r in results:
            print(
                f"    -> {r['label']}  "
                f"(qid={r['qid']}, coverage={r['coverage']}, avg_rating={r['avg_rating']:.2f})"
            )

    return results


# -------------------------------------------------------------------
# Quick evaluation on the 5 benchmark query sets you used before
# -------------------------------------------------------------------
TEST_QUERIES = {
    "musical_1": [
        "Singin' in the Rain",
        "Moulin Rouge",
    ],
    "fellini": [
        "La Dolce Vita",
        "The Voice of the Moon",
    ],
    "chicago_geisha_alice": [
        "Chicago",
        "Memoirs of a Geisha",
        "Alice in Wonderland",
    ],
    "forrest_lotr": [
        "Forrest Gump",
        "The Lord of the Rings: The Fellowship of the Ring",
    ],
    "japanese_kyoto": [
        "Twin Sisters of Kyoto",
    ],
}

for name, seeds in TEST_QUERIES.items():
    print("\n" + "=" * 70)
    print(f"Running graph_recommendation_and_or for: {name}")
    recs = graph_recommendation_and_or(seeds, top_k=10, print_debug=True)
    print("Top labels:", [r["label"] for r in recs])



Running graph_recommendation_and_or for: musical_1

Graph AND/OR recommendation for seeds: ["Singin' in the Rain", 'Moulin Rouge']
  Seed "Singin' in the Rain" -> uri=http://www.wikidata.org/entity/Q309153 qid=Q309153 label=Singin' in the Rain movie=True
  Seed 'Moulin Rouge' -> uri=http://www.wikidata.org/entity/Q1508611 qid=Q1508611 label=Moulin Rouge movie=True

=== Graph neighbors for seed QID Q309153 (http://www.wikidata.org/entity/Q309153) ===
Total raw rows: 50
Sample neighbors (up to 5):
  Q1062362  label='Love Songs'  ratings=[None]
  Q685245  label='The Fabulous Baker Boys'  ratings=[None]
  Q2291842  label="Who's That Girl"  ratings=[4.8]
  Q7560207  label='Something for the Boys'  ratings=[6.0]
  Q1214975  label='The Young Girls of Rochefort'  ratings=[7.7]

=== Graph neighbors for seed QID Q1508611 (http://www.wikidata.org/entity/Q1508611) ===
Total raw rows: 50
Sample neighbors (up to 5):
  Q313315  label='Tom Jones'  ratings=[None]
  Q1856597  label='Mary, Queen of Scot

In [35]:
import numpy as np

# ---------- Helper: run SPARQL and return list-of-dicts ----------
def run_sparql(query: str):
    """Execute a SPARQL SELECT on the local rdflib graph and return list of dicts."""
    res = graph.graph.query(query)
    out = []
    vars_ = [str(v) for v in res.vars]
    for row in res:
        d = {}
        for v in vars_:
            val = getattr(row, v, None)
            if val is not None:
                d[v] = str(val)
        out.append(d)
    return out


# ---------- Hybrid graph + embedding recommendation ----------
def graph_embedding_hybrid_recommendation(
    seed_titles,
    k_graph_per_seed: int = 200,
    top_k: int = 10,
    use_and: bool = True,
    print_debug: bool = True,
):
    """
    Hybrid recommender:
      1) Use graph neighbors (same director / same genre) as candidate set.
      2) Re-rank candidates using KG embeddings.
    """

    print("\n" + "=" * 70)
    print("Hybrid graph+embedding recommendation for seeds:", seed_titles)

    # 1. Link titles to movie URIs/QIDs
    seeds = []  # list of dicts: {title, uri, qid, label}
    for title in seed_titles:
        uri, lbl, is_movie = link_movie(title, must_be_movie=True)
        if uri is None or not is_movie:
            if print_debug:
                print(f"  Seed {title!r}: could not link to a movie (uri={uri}, is_movie={is_movie}). Skipping.")
            continue
        qid = uri.rsplit("/", 1)[-1]
        seeds.append({"title": title, "uri": uri, "qid": qid, "label": lbl})

    if not seeds:
        print("❌ No seeds could be linked to movies; aborting.")
        return []

    if print_debug:
        print("\nLinked seeds:")
        for s in seeds:
            print(f"  {s['title']!r} -> uri={s['uri']} qid={s['qid']} label={s['label']}")

    # 2. Graph neighbors for each seed
    neighbor_sets = []  # list of set(QID)
    candidates = {}     # qid -> info dict

    for s in seeds:
        uri = s["uri"]
        qid_seed = s["qid"]
        # Use existing composer and adjust LIMIT if needed
        query = composer.compose_graph_rec_query(uri)
        if k_graph_per_seed != 50:
            query = query.replace("LIMIT 50", f"LIMIT {k_graph_per_seed}")

        rows = run_sparql(query)
        qids_local = set()

        if print_debug:
            print(f"\n=== Graph neighbors for seed QID {qid_seed} ({uri}) ===")
            print(f"Total raw rows: {len(rows)}")

        for i, row in enumerate(rows):
            movie_uri = row.get("movie")
            if movie_uri is None:
                continue
            cqid = movie_uri.rsplit("/", 1)[-1]
            qids_local.add(cqid)

            rating_str = row.get("rating")
            rating = None
            if rating_str is not None:
                try:
                    rating = float(rating_str)
                except ValueError:
                    rating = None

            if cqid not in candidates:
                candidates[cqid] = {
                    "label": qid2label.get(cqid, cqid),
                    "ratings": [],
                    "coverage": 0,
                    "from_seeds": set(),
                }
            candidates[cqid]["coverage"] += 1
            candidates[cqid]["from_seeds"].add(qid_seed)
            if rating is not None:
                candidates[cqid]["ratings"].append(rating)

        neighbor_sets.append(qids_local)

        if print_debug and rows:
            print("Sample neighbors (up to 5):")
            for j, row in enumerate(rows[:5]):
                cqid = row["movie"].rsplit("/", 1)[-1]
                lab = qid2label.get(cqid, cqid)
                rating_str = row.get("rating")
                print(f"  {cqid}  label='{lab}'  ratings={[rating_str] if rating_str else [None]}")

    # Remove seeds themselves from candidates / neighbor_sets
    seed_qids = {s["qid"] for s in seeds}
    for sq in seed_qids:
        candidates.pop(sq, None)
    for sset in neighbor_sets:
        sset.difference_update(seed_qids)

    # 3. Decide active candidate set: AND vs OR
    and_qids = set.intersection(*neighbor_sets) if (use_and and neighbor_sets) else set()
    if print_debug:
        sizes = [len(s) for s in neighbor_sets]
        print("\n  Per-seed neighbor sizes:", sizes)
        print("  AND intersection size:", len(and_qids))

    if use_and and len(and_qids) > 0:
        active_qids = and_qids
        if print_debug:
            print("  Using AND semantics (common neighbors of all seeds).")
    else:
        active_qids = set().union(*neighbor_sets)
        if print_debug:
            print("  AND set empty or disabled, using OR-union.")

    active_candidates = {qid: candidates[qid] for qid in active_qids if qid in candidates}
    if not active_candidates:
        print("❌ No graph-based candidates found; aborting.")
        return []

    # 4. Prepare seed embeddings
    seed_emb_indices = [qid2idx[s["qid"]] for s in seeds if s["qid"] in qid2idx]
    if not seed_emb_indices:
        if print_debug:
            print("⚠️ No seed has an embedding; falling back to pure graph ranking.")
        # Fallback: just rank by coverage and avg_rating
        for info in active_candidates.values():
            if info["ratings"]:
                info["avg_rating"] = float(sum(info["ratings"]) / len(info["ratings"]))
            else:
                info["avg_rating"] = 0.0
            info["emb_dist"] = None
            info["has_emb"] = False
    else:
        seed_emb = E[seed_emb_indices]  # shape: (#seeds, dim)
        # 5. For each candidate, compute avg_rating and embedding distance
        for qid, info in active_candidates.items():
            # Rating aggregation
            if info["ratings"]:
                info["avg_rating"] = float(sum(info["ratings"]) / len(info["ratings"]))
            else:
                info["avg_rating"] = 0.0

            # Embedding distance
            if qid in qid2idx:
                idx = qid2idx[qid]
                v = E[idx]
                diffs = seed_emb - v
                dists = np.sum(diffs * diffs, axis=1)  # L2 squared per seed
                emb_dist = float(dists.mean())
                info["emb_dist"] = emb_dist
                info["has_emb"] = True
            else:
                info["emb_dist"] = None
                info["has_emb"] = False

    # 6. Sort candidates
    def sort_key(item):
        qid, info = item
        has_emb = info.get("has_emb", False)
        emb_dist = info["emb_dist"] if info["emb_dist"] is not None else 1e9
        coverage = info["coverage"]
        avg_rating = info["avg_rating"]
        label = info["label"]
        # (embedding available first, then closer in embedding space,
        #  then higher coverage, then higher rating, then label)
        return (0 if has_emb else 1, emb_dist, -coverage, -avg_rating, label)

    ranked = sorted(active_candidates.items(), key=sort_key)
    ranked = ranked[:top_k]

    # 7. Print final results
    print("\n  Final ranked hybrid recommendations:")
    for qid, info in ranked:
        print(
            f"    -> {info['label']}  (qid={qid}, "
            f"emb_dist={info['emb_dist']:.4f} if info['emb_dist'] is not None else 'None', "
            f"coverage={info['coverage']}, avg_rating={info['avg_rating']:.2f})"
        )

    top_labels = [info["label"] for qid, info in ranked]
    print("Top labels:", top_labels)
    return ranked


# ---------- Quick tests with the same 5 scenarios ----------
tests = {
    "musical_1": ["Singin' in the Rain", "Moulin Rouge"],
    "fellini": ["La Dolce Vita", "The Voice of the Moon"],
    "chicago_geisha_alice": ["Chicago", "Memoirs of a Geisha", "Alice in Wonderland"],
    "forrest_lotr": ["Forrest Gump", "The Lord of the Rings: The Fellowship of the Ring"],
    "japanese_kyoto": ["Twin Sisters of Kyoto"],
}

for name, seeds in tests.items():
    print("\n" + "=" * 70)
    print(f"Running hybrid graph+embedding recommendation for: {name}")
    graph_embedding_hybrid_recommendation(
        seeds,
        k_graph_per_seed=200,
        top_k=10,
        use_and=True,
        print_debug=True,
    )



Running hybrid graph+embedding recommendation for: musical_1

Hybrid graph+embedding recommendation for seeds: ["Singin' in the Rain", 'Moulin Rouge']

Linked seeds:
  "Singin' in the Rain" -> uri=http://www.wikidata.org/entity/Q309153 qid=Q309153 label=Singin' in the Rain
  'Moulin Rouge' -> uri=http://www.wikidata.org/entity/Q1508611 qid=Q1508611 label=Moulin Rouge

=== Graph neighbors for seed QID Q309153 (http://www.wikidata.org/entity/Q309153) ===
Total raw rows: 200
Sample neighbors (up to 5):
  Q1062362  label='Love Songs'  ratings=[None]
  Q685245  label='The Fabulous Baker Boys'  ratings=[None]
  Q2291842  label='Who's That Girl'  ratings=['4.8']
  Q7560207  label='Something for the Boys'  ratings=['6.0']
  Q1214975  label='The Young Girls of Rochefort'  ratings=['7.7']

=== Graph neighbors for seed QID Q1508611 (http://www.wikidata.org/entity/Q1508611) ===
Total raw rows: 200
Sample neighbors (up to 5):
  Q313315  label='Tom Jones'  ratings=[None]
  Q1856597  label='Mary, Qu

# A Useable Recommendation

In [36]:
import numpy as np

def run_sparql(query: str):
    """Execute a SPARQL SELECT on the local rdflib graph and return list of dicts."""
    res = graph.graph.query(query)
    out = []
    vars_ = [str(v) for v in res.vars]
    for row in res:
        d = {}
        for v in vars_:
            val = getattr(row, v, None)
            if val is not None:
                d[v] = str(val)
        out.append(d)
    return out


def graph_embedding_hybrid_recommendation(
    seed_titles,
    k_graph_per_seed: int = 200,
    top_k: int = 10,
    use_and: bool = True,
    print_debug: bool = True,
):
    """
    Hybrid recommender:
      1) Use graph neighbors (same director / same genre) as candidate set.
      2) Rank primarily by graph structure (coverage, rating),
         and only secondarily by embeddings.
    """

    print("\n" + "=" * 70)
    print("Hybrid graph+embedding recommendation for seeds:", seed_titles)

    # 1. Link titles to movie URIs/QIDs
    seeds = []
    for title in seed_titles:
        uri, lbl, is_movie = link_movie(title, must_be_movie=True)
        if uri is None or not is_movie:
            if print_debug:
                print(f"  Seed {title!r}: could not link to a movie (uri={uri}, is_movie={is_movie}). Skipping.")
            continue
        qid = uri.rsplit("/", 1)[-1]
        seeds.append({"title": title, "uri": uri, "qid": qid, "label": lbl})

    if not seeds:
        print("❌ No seeds could be linked to movies; aborting.")
        return []

    if print_debug:
        print("\nLinked seeds:")
        for s in seeds:
            print(f"  {s['title']!r} -> uri={s['uri']} qid={s['qid']} label={s['label']}")

    # 2. Graph neighbors per seed
    neighbor_sets = []
    candidates = {}  # qid -> info dict

    for s in seeds:
        uri = s["uri"]
        qid_seed = s["qid"]

        query = composer.compose_graph_rec_query(uri)
        if k_graph_per_seed != 50:
            query = query.replace("LIMIT 50", f"LIMIT {k_graph_per_seed}")

        rows = run_sparql(query)
        qids_local = set()

        if print_debug:
            print(f"\n=== Graph neighbors for seed QID {qid_seed} ({uri}) ===")
            print(f"Total raw rows: {len(rows)}")

        for i, row in enumerate(rows):
            movie_uri = row.get("movie")
            if movie_uri is None:
                continue
            cqid = movie_uri.rsplit("/", 1)[-1]
            qids_local.add(cqid)

            rating_str = row.get("rating")
            rating = None
            if rating_str is not None:
                try:
                    rating = float(rating_str)
                except ValueError:
                    rating = None

            if cqid not in candidates:
                candidates[cqid] = {
                    "label": qid2label.get(cqid, cqid),
                    "ratings": [],
                    "coverage": 0,
                    "from_seeds": set(),
                }
            candidates[cqid]["coverage"] += 1
            candidates[cqid]["from_seeds"].add(qid_seed)
            if rating is not None:
                candidates[cqid]["ratings"].append(rating)

        neighbor_sets.append(qids_local)

        if print_debug and rows:
            print("Sample neighbors (up to 5):")
            for row in rows[:5]:
                cqid = row["movie"].rsplit("/", 1)[-1]
                lab = qid2label.get(cqid, cqid)
                rating_str = row.get("rating")
                print(f"  {cqid}  label='{lab}'  ratings={[rating_str] if rating_str else [None]}")

    # Remove seeds from candidates and neighbor sets
    seed_qids = {s["qid"] for s in seeds}
    for sq in seed_qids:
        candidates.pop(sq, None)
    for sset in neighbor_sets:
        sset.difference_update(seed_qids)

    # 3. Decide active candidate set: AND vs OR
    and_qids = set.intersection(*neighbor_sets) if (use_and and neighbor_sets) else set()
    if print_debug:
        sizes = [len(s) for s in neighbor_sets]
        print("\n  Per-seed neighbor sizes:", sizes)
        print("  AND intersection size:", len(and_qids))

    if use_and and len(and_qids) > 0:
        active_qids = and_qids
        if print_debug:
            print("  Using AND semantics (common neighbors across all seeds).")
    else:
        active_qids = set().union(*neighbor_sets)
        if print_debug:
            print("  AND set empty or disabled, using OR-union.")

    active_candidates = {qid: candidates[qid] for qid in active_qids if qid in candidates}
    if not active_candidates:
        print("❌ No graph-based candidates found; aborting.")
        return []

    # 4. Prepare seed embeddings
    seed_emb_indices = [qid2idx[s["qid"]] for s in seeds if s["qid"] in qid2idx]
    if not seed_emb_indices:
        if print_debug:
            print("⚠️ No seed has an embedding; falling back to pure graph ranking.")
        for info in active_candidates.values():
            if info["ratings"]:
                info["avg_rating"] = float(sum(info["ratings"]) / len(info["ratings"]))
            else:
                info["avg_rating"] = 0.0
            info["emb_dist"] = None
            info["has_emb"] = False
    else:
        seed_emb = E[seed_emb_indices]
        for qid, info in active_candidates.items():
            # rating
            if info["ratings"]:
                info["avg_rating"] = float(sum(info["ratings"]) / len(info["ratings"]))
            else:
                info["avg_rating"] = 0.0

            # embedding dist
            if qid in qid2idx:
                idx = qid2idx[qid]
                v = E[idx]
                diffs = seed_emb - v
                dists = np.sum(diffs * diffs, axis=1)
                emb_dist = float(dists.mean())
                info["emb_dist"] = emb_dist
                info["has_emb"] = True
            else:
                info["emb_dist"] = None
                info["has_emb"] = False

    # 5. New sort rule: graph first, embedding later
    def sort_key(item):
        qid, info = item
        has_emb = info.get("has_emb", False)
        emb_dist = info["emb_dist"] if info["emb_dist"] is not None else 1e9
        coverage = info["coverage"]
        avg_rating = info["avg_rating"]
        label = info["label"]

        return (
            0 if has_emb else 1,  # still略微偏好有 embedding 的
            -coverage,            # 覆盖多 seed 的更前 ≈ AND
            -avg_rating,          # 高评分更前
            emb_dist,             # 最后再看 embedding 距离
            label,
        )

    ranked = sorted(active_candidates.items(), key=sort_key)[:top_k]

    # 6. Pretty print
    print("\n  Final ranked hybrid recommendations:")
    for qid, info in ranked:
        emb_str = f"{info['emb_dist']:.4f}" if info["emb_dist"] is not None else "None"
        print(
            f"    -> {info['label']}  "
            f"(qid={qid}, emb_dist={emb_str}, "
            f"coverage={info['coverage']}, avg_rating={info['avg_rating']:.2f})"
        )

    top_labels = [info["label"] for qid, info in ranked]
    print("Top labels:", top_labels)
    return ranked


# Quick sanity check with the same scenarios
tests = {
    "musical_1": ["Singin' in the Rain", "Moulin Rouge"],
    "fellini": ["La Dolce Vita", "The Voice of the Moon"],
    "chicago_geisha_alice": ["Chicago", "Memoirs of a Geisha", "Alice in Wonderland"],
    "forrest_lotr": ["Forrest Gump", "The Lord of the Rings: The Fellowship of the Ring"],
    "japanese_kyoto": ["Twin Sisters of Kyoto"],
}

for name, seeds in tests.items():
    print("\n" + "=" * 70)
    print(f"Running hybrid graph+embedding recommendation for: {name}")
    graph_embedding_hybrid_recommendation(
        seeds,
        k_graph_per_seed=200,
        top_k=10,
        use_and=True,
        print_debug=True,
    )



Running hybrid graph+embedding recommendation for: musical_1

Hybrid graph+embedding recommendation for seeds: ["Singin' in the Rain", 'Moulin Rouge']

Linked seeds:
  "Singin' in the Rain" -> uri=http://www.wikidata.org/entity/Q309153 qid=Q309153 label=Singin' in the Rain
  'Moulin Rouge' -> uri=http://www.wikidata.org/entity/Q1508611 qid=Q1508611 label=Moulin Rouge

=== Graph neighbors for seed QID Q309153 (http://www.wikidata.org/entity/Q309153) ===
Total raw rows: 200
Sample neighbors (up to 5):
  Q1062362  label='Love Songs'  ratings=[None]
  Q685245  label='The Fabulous Baker Boys'  ratings=[None]
  Q2291842  label='Who's That Girl'  ratings=['4.8']
  Q7560207  label='Something for the Boys'  ratings=['6.0']
  Q1214975  label='The Young Girls of Rochefort'  ratings=['7.7']

=== Graph neighbors for seed QID Q1508611 (http://www.wikidata.org/entity/Q1508611) ===
Total raw rows: 200
Sample neighbors (up to 5):
  Q313315  label='Tom Jones'  ratings=[None]
  Q1856597  label='Mary, Qu

# Generate wdt: of films

In [38]:
# %%
"""
Inspect which wdt: predicates are actually used on movies in our local KG.

- Restrict to entities with `wdt:P31 wd:Q11424` (instance of: film).
- For these movies, count how often each direct property (?movie ?p ?o) appears.
- Only keep wdt: predicates (http://www.wikidata.org/prop/direct/Pxxx).
- Save a TSV file `movie_predicate_stats.tsv` with columns:
    - predicate_uri
    - pid (e.g. P57, P136, ...)
    - label (if available in the KG)
    - count
"""

import re
import pandas as pd

from agent.graph_executor import GraphExecutor
from agent.constants import PREFIXES

# 如果之前的 notebook 里已经创建了 graph，就直接用
# 否则用 Config 手动加载一次（注意这里的 cfg 属性名要和你自己的 Config 对上）
try:
    graph  # noqa: F821
except NameError:
    from agent.config import Config
    cfg = Config()
    # ⚠️ 这里的属性名可能是 cfg.graph_path / cfg.GRAPH_PATH / cfg.kg_path 等，
    # 请根据你的 Config 实际字段改一下
    KG_PATH = getattr(cfg, "graph_path", None) or getattr(cfg, "GRAPH_PATH", None) or getattr(cfg, "kg_path", None)
    if KG_PATH is None:
        raise RuntimeError("Cannot find KG path on Config; please set KG_PATH manually here.")
    graph = GraphExecutor(KG_PATH)

# 2) SPARQL：统计电影上的 wdt: 属性
query = f"""
{PREFIXES}
SELECT ?p (SAMPLE(?plabel) AS ?plabel) (COUNT(*) AS ?c)
WHERE {{
  ?movie wdt:P31 wd:Q11424 .
  ?movie ?p ?o .
  # 只保留 wdt: direct properties
  FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/")) .
  OPTIONAL {{
    ?p rdfs:label ?plabel .
    FILTER(LANG(?plabel) = "en")
  }}
}}
GROUP BY ?p
ORDER BY DESC(?c)
LIMIT 200
"""

print("Running predicate statistics query on movie subset...")
res = graph.graph.query(query)

rows = []
for row in res:
    p_uri = str(row["p"])
    label = str(row["plabel"]) if row["plabel"] is not None else None
    count = int(row["c"])

    # 从 URI 里提取 P-id（例如 P57）
    m = re.search(r"/prop/direct/(P\\d+)$", p_uri)
    pid = m.group(1) if m else None

    rows.append(
        {
            "predicate_uri": p_uri,
            "pid": pid,
            "label": label,
            "count": count,
        }
    )

df = pd.DataFrame(rows).sort_values("count", ascending=False)

print(f"\nTotal distinct wdt: predicates on movies (sampled): {len(df)}\n")
print("Top 20 predicates on movies:")
print(df.head(20).to_string(index=False))

# 3) 保存 TSV，之后可以在 VS Code / Excel 里慢慢看
output_path = "movie_predicate_stats.tsv"
df.to_csv(output_path, sep="\t", index=False)
print(f"\nSaved predicate stats to: {output_path}")

# 4) 把我们关注的一些属性点名看看是否在数据里
interesting = {
    "P57":  "director",
    "P136": "genre",
    "P495": "country of origin",
    "P364": "language of work or name",
    "P86":  "composer",
    "P161": "cast member",
    "P166": "award received",
}

print("\nInteresting attributes (if present in this sample):")
for pid, desc in interesting.items():
    hit = df[df["pid"] == pid]
    if not hit.empty:
        row = hit.iloc[0]
        print(
            f"  {pid:>4} ({desc:25s})  "
            f"count={row['count']:<6d}  "
            f"label={row['label']}"
        )
    else:
        print(f"  {pid:>4} ({desc:25s})  NOT FOUND in top 200 predicates sample")


Running predicate statistics query on movie subset...

Total distinct wdt: predicates on movies (sampled): 186

Top 20 predicates on movies:
                            predicate_uri  pid                                label  count
 http://www.wikidata.org/prop/direct/P161 None                          cast member 159023
 http://www.wikidata.org/prop/direct/P136 None                                genre  32162
  http://www.wikidata.org/prop/direct/P58 None                         screenwriter  18296
 http://www.wikidata.org/prop/direct/P750 None                       distributed by  18018
http://www.wikidata.org/prop/direct/P1411 None                        nominated for  17781
 http://www.wikidata.org/prop/direct/P495 None                    country of origin  16521
 http://www.wikidata.org/prop/direct/P162 None                             producer  15561
 http://www.wikidata.org/prop/direct/P364 None original language of film or TV show  13773
  http://www.wikidata.org/prop/direct/P5

In [39]:
# %%
"""
Fix predicate PIDs and prototype attribute-based recommendations for:
1) "in Japanese" + seed film  -> P364 = Japanese
2) "biographical movies" + "Meryl Streep" -> P136 = biographical film, P161 = Meryl Streep
"""

import re
import math
import pandas as pd

from agent.graph_executor import GraphExecutor
from agent.constants import PREFIXES

# --- 0. Ensure we have a graph and qid2label/qid2idx from previous cells ---

try:
    graph  # noqa: F821
except NameError:
    from agent.config import Config
    cfg = Config()
    KG_PATH = getattr(cfg, "graph_path", None) or getattr(cfg, "GRAPH_PATH", None) or getattr(cfg, "kg_path", None)
    if KG_PATH is None:
        raise RuntimeError("Cannot find KG path on Config; please set KG_PATH manually here.")
    graph = GraphExecutor(KG_PATH)

try:
    qid2label  # noqa: F821
    qid2idx    # noqa: F821
except NameError:
    raise RuntimeError("qid2label / qid2idx not found. Please run the embedding–QID mapping cell first.")


# --- 1. Fix predicate PIDs from the TSV we just wrote earlier ---

pred_df = pd.read_csv("movie_predicate_stats.tsv", sep="\t")

# 无论之前那一列是什么，重新计算一次 pid，避免之前 regex 的 bug
pred_df["pid"] = pred_df["predicate_uri"].str.extract(r"/prop/direct/(P\d+)$")

print("Fixed pid extraction. Sample:")
print(pred_df.head(10).to_string(index=False))

# 再看一下我们关心的几个属性是否在表里
interesting = {
    "P57":  "director",
    "P136": "genre",
    "P495": "country of origin",
    "P364": "original language of work",
    "P86":  "composer",
    "P161": "cast member",
    "P166": "award received",
}

print("\nInteresting predicates (after pid fix):")
for pid, desc in interesting.items():
    hit = pred_df[pred_df["pid"] == pid]
    if not hit.empty:
        row = hit.iloc[0]
        print(
            f"  {pid:>4} ({desc:30s})  "
            f"count={int(row['count']):<6d}  "
            f"label={row['label']}"
        )
    else:
        print(f"  {pid:>4} ({desc:30s})  NOT FOUND in stats (maybe very rare or absent).")


# --- 2. Small helpers: label <-> QID search on our KG label map ---


def find_qids_by_label_substring(substr, limit=10):
    """在 qid2label 里用子串匹配，返回 [(qid, label)]。大小写不敏感。"""
    sub = substr.lower()
    hits = []
    for qid, lbl in qid2label.items():
        if sub in lbl.lower():
            hits.append((qid, lbl))
            if len(hits) >= limit:
                break
    return hits


def find_unique_qid_exact(label_text):
    """简单 exact match，返回唯一 qid，否则 None。"""
    target = label_text.strip().lower()
    matches = [qid for qid, lbl in qid2label.items() if lbl.lower() == target]
    if len(matches) == 1:
        return matches[0]
    return None


# --- 3. Attribute-based recommendation 1: "in Japanese" + Twin Sisters of Kyoto ---


def recommend_in_japanese_like_twin_sisters(top_k=15, debug=True):
    question = "What other movies in Japanese do you recommend? I liked Twin Sisters of Kyoto."
    seed_title = "Twin Sisters of Kyoto"

    # 找 seed 的 QID（用 exact 匹配优先）
    seed_qid = find_unique_qid_exact(seed_title)
    if seed_qid is None:
        # fallback: 子串匹配
        cand = find_qids_by_label_substring(seed_title, limit=3)
        if not cand:
            raise RuntimeError(f"Could not find QID for seed title: {seed_title}")
        seed_qid = cand[0][0]

    # 找语言 Japanese language 的 QID
    # 先试试 exact "Japanese" 不带 language
    lang_hits = [
        (qid, lbl)
        for qid, lbl in qid2label.items()
        if "japanese" in lbl.lower() and "language" in lbl.lower()
    ]
    if not lang_hits:
        raise RuntimeError("Could not find a QID for 'Japanese language' in qid2label.")
    lang_qid, lang_label = lang_hits[0]

    if debug:
        print("=== Attribute-based Japanese-language rec ===")
        print(f"Question: {question}")
        print(f"Seed: {seed_title!r} -> {seed_qid} [{qid2label.get(seed_qid, '?')}]")
        print(f"Language 'Japanese' -> {lang_qid} [{lang_label}]")

    # SPARQL：所有原始语言为 Japanese 的电影，排除种子本身，按 ddis:rating 排序
    query = f"""
    {PREFIXES}
    SELECT DISTINCT ?movie ?rating WHERE {{
      ?movie wdt:P31 wd:Q11424 .
      ?movie wdt:P364 wd:{lang_qid} .
      FILTER(?movie != wd:{seed_qid})
      OPTIONAL {{ ?movie ddis:rating ?rating . }}
    }}
    """

    res = graph.graph.query(query)

    candidates = []
    for row in res:
        uri = str(row["movie"])
        m = re.search(r"/entity/(Q\d+)$", uri)
        if not m:
            continue
        qid = m.group(1)
        rating = None
        if row.get("rating") is not None:
            try:
                rating = float(row["rating"])
            except Exception:
                rating = None
        candidates.append(
            {
                "qid": qid,
                "uri": uri,
                "label": qid2label.get(qid, None),
                "rating": rating,
            }
        )

    # 按 rating 降序排序，未标 rating 的放后
    def sort_key(it):
        r = it["rating"]
        return (0 if r is not None else 1, -(r if r is not None else 0.0))

    candidates.sort(key=sort_key)

    if debug:
        print(f"\nTotal Japanese-language movie candidates (excluding seed): {len(candidates)}")
        print(f"Top {top_k} by rating:")
        for i, item in enumerate(candidates[:top_k], start=1):
            print(
                f"  #{i:2d} {item['label'] or item['qid']} "
                f"(qid={item['qid']}, rating={item['rating']})"
            )

    return candidates[:top_k]


# --- 4. Attribute-based recommendation 2: biographical movies + Meryl Streep ---


def recommend_biographical_meryl_streep(top_k=15, debug=True):
    question = "Can you recommend some biographical movies given that I like Meryl Streep?"

    # 找 Meryl Streep 的 QID
    meryl_qid = find_unique_qid_exact("Meryl Streep")
    if meryl_qid is None:
        hits = find_qids_by_label_substring("Meryl Streep", limit=3)
        if not hits:
            raise RuntimeError("Could not find QID for 'Meryl Streep'.")
        meryl_qid = hits[0][0]

    # 找 "biographical film" 这个 genre 的 QID
    bio_hits_exact = [
        (qid, lbl)
        for qid, lbl in qid2label.items()
        if lbl.lower() == "biographical film"
    ]
    if bio_hits_exact:
        bio_qid, bio_label = bio_hits_exact[0]
    else:
        bio_hits = find_qids_by_label_substring("biographical film", limit=3)
        if not bio_hits:
            raise RuntimeError("Could not find a QID for 'biographical film'.")
        bio_qid, bio_label = bio_hits[0]

    if debug:
        print("\n=== Attribute-based biographical Meryl Streep rec ===")
        print(f"Question: {question}")
        print(f"Meryl Streep -> {meryl_qid} [{qid2label.get(meryl_qid, '?')}]")
        print(f"Biographical genre -> {bio_qid} [{bio_label}]")

    # 4.1 先求交集：同时满足「Meryl Streep 演」+「genre = biographical film」
    query_and = f"""
    {PREFIXES}
    SELECT DISTINCT ?movie ?rating WHERE {{
      ?movie wdt:P31 wd:Q11424 .
      ?movie wdt:P161 wd:{meryl_qid} .
      ?movie wdt:P136 wd:{bio_qid} .
      OPTIONAL {{ ?movie ddis:rating ?rating . }}
    }}
    """

    res_and = list(graph.graph.query(query_and))

    def rows_to_candidates(rows):
        out = []
        for row in rows:
            uri = str(row["movie"])
            m = re.search(r"/entity/(Q\d+)$", uri)
            if not m:
                continue
            qid = m.group(1)
            rating = None
            if row.get("rating") is not None:
                try:
                    rating = float(row["rating"])
                except Exception:
                    rating = None
            out.append(
                {
                    "qid": qid,
                    "uri": uri,
                    "label": qid2label.get(qid, None),
                    "rating": rating,
                }
            )
        return out

    cand_and = rows_to_candidates(res_and)

    if debug:
        print(f"\nIntersection (Meryl Streep & biographical): {len(cand_and)} films")

    # 如果交集有结果，就用交集；否则 fallback 到 OR：Meryl Streep 的电影 ∪ 所有传记片
    if cand_and:
        candidates = cand_and
    else:
        if debug:
            print("Intersection empty, falling back to OR (union) strategy.")

        query_or = f"""
        {PREFIXES}
        SELECT DISTINCT ?movie ?rating WHERE {{
          ?movie wdt:P31 wd:Q11424 .
          {{
            ?movie wdt:P161 wd:{meryl_qid} .
          }} UNION {{
            ?movie wdt:P136 wd:{bio_qid} .
          }}
          OPTIONAL {{ ?movie ddis:rating ?rating . }}
        }}
        """
        res_or = list(graph.graph.query(query_or))
        candidates = rows_to_candidates(res_or)

    # 按 rating 排序
    def sort_key(it):
        r = it["rating"]
        return (0 if r is not None else 1, -(r if r is not None else 0.0))

    candidates.sort(key=sort_key)

    if debug:
        print(f"Total candidate films: {len(candidates)}")
        print(f"Top {top_k}:")
        for i, item in enumerate(candidates[:top_k], start=1):
            print(
                f"  #{i:2d} {item['label'] or item['qid']} "
                f"(qid={item['qid']}, rating={item['rating']})"
            )

    return candidates[:top_k]


# --- 5. Run both tests once in the notebook ---

print("\n======================================================================")
print("Running attribute-based recommendation tests\n")

jp_recs = recommend_in_japanese_like_twin_sisters(top_k=15, debug=True)

print("\n----------------------------------------------------------------------")

meryl_recs = recommend_biographical_meryl_streep(top_k=15, debug=True)


Fixed pid extraction. Sample:
                            predicate_uri   pid                                label  count
 http://www.wikidata.org/prop/direct/P161  P161                          cast member 159023
 http://www.wikidata.org/prop/direct/P136  P136                                genre  32162
  http://www.wikidata.org/prop/direct/P58   P58                         screenwriter  18296
 http://www.wikidata.org/prop/direct/P750  P750                       distributed by  18018
http://www.wikidata.org/prop/direct/P1411 P1411                        nominated for  17781
 http://www.wikidata.org/prop/direct/P495  P495                    country of origin  16521
 http://www.wikidata.org/prop/direct/P162  P162                             producer  15561
 http://www.wikidata.org/prop/direct/P364  P364 original language of film or TV show  13773
  http://www.wikidata.org/prop/direct/P57   P57                             director  13168
  http://www.wikidata.org/prop/direct/P31   P31   

RuntimeError: Could not find a QID for 'Japanese language' in qid2label.

In [40]:
# %%
import re

# --- helper: 从 qid2label 或 SPARQL 中找到 Japanese language 的 QID ---


def get_japanese_language_qid():
    """
    先在 qid2label 里找 'Japanese language' 或 'Japanese'，
    找不到再用 SPARQL 按 rdfs:label 查。
    返回 (qid, label_str)
    """
    # 1) try qid2label (万一里边刚好有语言实体就省一次 SPARQL)
    for qid, lbl in qid2label.items():
        lower = lbl.lower()
        if lower == "japanese language" or lower == "japanese":
            return qid, lbl

    # 2) fallback: 用 SPARQL 在 KG 里按 label 查
    query = f"""
    {PREFIXES}
    SELECT ?lang ?lbl WHERE {{
      ?lang rdfs:label ?lbl .
      FILTER(
        (?lbl = "Japanese language"@en) ||
        (?lbl = "Japanese"@en)
      )
    }} LIMIT 5
    """
    rows = list(graph.graph.query(query))
    if not rows:
        raise RuntimeError("Could not resolve 'Japanese language' via SPARQL.")

    row = rows[0]
    uri = str(row["lang"])
    m = re.search(r"/entity/(Q\\d+)$", uri)
    if not m:
        raise RuntimeError(f"Unexpected lang URI: {uri}")
    qid = m.group(1)
    lbl = str(row["lbl"])
    return qid, lbl


# --- 覆盖之前的 recommend_in_japanese_like_twin_sisters ---


def recommend_in_japanese_like_twin_sisters(top_k=15, debug=True):
    question = "What other movies in Japanese do you recommend? I liked Twin Sisters of Kyoto."
    seed_title = "Twin Sisters of Kyoto"

    # 找 seed 的 QID（优先 exact，失败再 substring）
    seed_qid = find_unique_qid_exact(seed_title)
    if seed_qid is None:
        cand = find_qids_by_label_substring(seed_title, limit=3)
        if not cand:
            raise RuntimeError(f"Could not find QID for seed title: {seed_title}")
        seed_qid = cand[0][0]

    # 通过 helper 拿 Japanese language 的 QID
    lang_qid, lang_label = get_japanese_language_qid()

    if debug:
        print("=== Attribute-based Japanese-language rec ===")
        print(f"Question: {question}")
        print(f"Seed: {seed_title!r} -> {seed_qid} [{qid2label.get(seed_qid, '?')}]")
        print(f"Language 'Japanese' -> {lang_qid} [{lang_label}]")

    # 所有原始语言为 Japanese 的电影，排除种子
    query = f"""
    {PREFIXES}
    SELECT DISTINCT ?movie ?rating WHERE {{
      ?movie wdt:P31 wd:Q11424 .
      ?movie wdt:P364 wd:{lang_qid} .
      FILTER(?movie != wd:{seed_qid})
      OPTIONAL {{ ?movie ddis:rating ?rating . }}
    }}
    """
    res = graph.graph.query(query)

    candidates = []
    for row in res:
        uri = str(row["movie"])
        m = re.search(r"/entity/(Q\\d+)$", uri)
        if not m:
            continue
        qid = m.group(1)
        rating = None
        # ResultRow 支持 get，我们跟之前保持一致
        if row.get("rating") is not None:
            try:
                rating = float(row["rating"])
            except Exception:
                rating = None
        candidates.append(
            {
                "qid": qid,
                "uri": uri,
                "label": qid2label.get(qid, None),
                "rating": rating,
            }
        )

    # rating 高的在前，没有 rating 的排在后面
    def sort_key(it):
        r = it["rating"]
        return (0 if r is not None else 1, -(r if r is not None else 0.0))

    candidates.sort(key=sort_key)

    if debug:
        print(f"\nTotal Japanese-language movie candidates (excluding seed): {len(candidates)}")
        print(f"Top {top_k} by rating:")
        for i, item in enumerate(candidates[:top_k], start=1):
            print(
                f"  #{i:2d} {item['label'] or item['qid']} "
                f"(qid={item['qid']}, rating={item['rating']})"
            )

    return candidates[:top_k]


# --- 重新跑两个测试：Japanese + Meryl Streep 传记片 ---


print("\n======================================================================")
print("Re-running attribute-based recommendation tests with Japanese-language fix\n")

jp_recs = recommend_in_japanese_like_twin_sisters(top_k=15, debug=True)

print("\n----------------------------------------------------------------------")

meryl_recs = recommend_biographical_meryl_streep(top_k=15, debug=True)



Re-running attribute-based recommendation tests with Japanese-language fix

=== Attribute-based Japanese-language rec ===
Question: What other movies in Japanese do you recommend? I liked Twin Sisters of Kyoto.
Seed: 'Twin Sisters of Kyoto' -> Q1458080 [Twin Sisters of Kyoto]
Language 'Japanese' -> Q5287 [Japanese]

Total Japanese-language movie candidates (excluding seed): 0
Top 15 by rating:

----------------------------------------------------------------------

=== Attribute-based biographical Meryl Streep rec ===
Question: Can you recommend some biographical movies given that I like Meryl Streep?
Meryl Streep -> Q873 [Meryl Streep]
Biographical genre -> Q645928 [biographical film]

Intersection (Meryl Streep & biographical): 9 films
Total candidate films: 9
Top 15:
  # 1 Evil Angels (qid=Q1249239, rating=6.9)
  # 2 Florence Foster Jenkins (qid=Q19955874, rating=None)
  # 3 The Iron Lady (qid=Q269810, rating=None)
  # 4 Julie & Julia (qid=Q380848, rating=None)
  # 5 The Post (qid=

In [43]:
# %%
import re

# ---------- 统一的 URI -> QID 提取函数 ----------

def extract_qid_from_uri(uri: str):
    """
    从类似 'http://www.wikidata.org/entity/Q1458080' 里提取 'Q1458080'。
    匹配不到返回 None。
    """
    m = re.search(r"/entity/(Q\d+)$", uri)
    return m.group(1) if m else None


# ---------- 更新：按 label 子串找实体 QID ----------

def get_entity_qid_by_label_substring(name_substr, limit=5, raise_if_missing=False):
    key = name_substr.lower()

    # 1) 先在 qid2label 里找
    for qid, lbl in qid2label.items():
        if key in lbl.lower():
            return qid, lbl

    # 2) 再用 SPARQL 找
    query = f"""
    {PREFIXES}
    SELECT ?ent ?lbl WHERE {{
      ?ent rdfs:label ?lbl .
      FILTER(CONTAINS(LCASE(STR(?lbl)), "{key}"))
    }} LIMIT {limit}
    """
    try:
        rows = list(graph.graph.query(query))
    except Exception:
        if raise_if_missing:
            raise
        return None, None

    if not rows:
        if raise_if_missing:
            raise RuntimeError(f"Could not resolve entity by label substring: {name_substr!r}")
        return None, None

    row = rows[0]
    uri = str(row["ent"])
    qid = extract_qid_from_uri(uri)
    if not qid:
        if raise_if_missing:
            raise RuntimeError(f"Unexpected entity URI: {uri}")
        return None, None

    lbl = str(row["lbl"])
    return qid, lbl


# ---------- Japanese language QID ----------

def get_japanese_language_qid():
    # 精确 "Japanese language"
    for qid, lbl in qid2label.items():
        if lbl.lower() == "japanese language":
            return qid, lbl

    # 精确 "Japanese"
    for qid, lbl in qid2label.items():
        if lbl.lower() == "japanese":
            return qid, lbl

    # 子串 fallback
    qid, lbl = get_entity_qid_by_label_substring("Japanese language", raise_if_missing=False)
    if qid is not None:
        return qid, lbl

    qid, lbl = get_entity_qid_by_label_substring("Japanese", raise_if_missing=False)
    if qid is not None:
        return qid, lbl

    # 实在没有就返回 None
    return None, None


# ---------- Takemitsu QID ----------

def get_takemitsu_qid():
    qid, lbl = get_entity_qid_by_label_substring("Takemitsu", raise_if_missing=False)
    return qid, lbl


# ---------- Japanese 问题：语言 + 作曲家，两路 + fallback ----------

def recommend_in_japanese_like_twin_sisters(top_k=15, debug=True):
    question = "What other movies in Japanese do you recommend? I liked Twin Sisters of Kyoto."
    seed_title = "Twin Sisters of Kyoto"

    # 找 seed
    seed_qid = find_unique_qid_exact(seed_title)
    if seed_qid is None:
        cand = find_qids_by_label_substring(seed_title, limit=3)
        if not cand:
            raise RuntimeError(f"Could not find QID for seed title: {seed_title}")
        seed_qid = cand[0][0]

    # 语言 & 作曲家
    lang_qid, lang_label = get_japanese_language_qid()
    tak_qid, tak_label   = get_takemitsu_qid()

    if debug:
        print("=== Attribute-based Japanese-language / Takemitsu rec ===")
        print(f"Question: {question}")
        print(f"Seed: {seed_title!r} -> {seed_qid} [{qid2label.get(seed_qid, '?')}]")
        if lang_qid is not None:
            print(f"Language 'Japanese' -> {lang_qid} [{lang_label}]")
        else:
            print("Language 'Japanese' -> NOT FOUND, skip language filter.")
        if tak_qid is not None:
            print(f"Composer 'Toru/Tōru Takemitsu' -> {tak_qid} [{tak_label}]")
        else:
            print("Composer 'Takemitsu' -> NOT FOUND, skip composer filter.")

    candidates_by_qid = {}

    # --- (1) 按语言 ---
    if lang_qid is not None:
        lang_query = f"""
        {PREFIXES}
        SELECT DISTINCT ?movie ?rating WHERE {{
          ?movie wdt:P31 wd:Q11424 .
          ?movie wdt:P364 wd:{lang_qid} .
          FILTER(?movie != wd:{seed_qid})
          OPTIONAL {{ ?movie ddis:rating ?rating . }}
        }}
        """
        lang_rows = list(graph.graph.query(lang_query))
        if debug:
            print(f"\nLanguage-based candidates (original language = Japanese): {len(lang_rows)}")

        for row in lang_rows:
            uri = str(row["movie"])
            qid = extract_qid_from_uri(uri)
            if not qid:
                continue

            rating = None
            r_term = row.get("rating")
            if r_term is not None:
                try:
                    rating = float(r_term)
                except Exception:
                    rating = None

            if qid not in candidates_by_qid:
                candidates_by_qid[qid] = {
                    "qid": qid,
                    "uri": uri,
                    "label": qid2label.get(qid, None),
                    "rating": rating,
                    "sources": set(["lang"]),
                }
            else:
                item = candidates_by_qid[qid]
                item["sources"].add("lang")
                if item["rating"] is None and rating is not None:
                    item["rating"] = rating
    else:
        if debug:
            print("\n[Info] No Japanese-language QID; language-based branch disabled.")

    # --- (2) 按作曲家 Takemitsu ---
    if tak_qid is not None:
        comp_query = f"""
        {PREFIXES}
        SELECT DISTINCT ?movie ?rating WHERE {{
          ?movie wdt:P31 wd:Q11424 .
          ?movie wdt:P86 wd:{tak_qid} .
          FILTER(?movie != wd:{seed_qid})
          OPTIONAL {{ ?movie ddis:rating ?rating . }}
        }}
        """
        comp_rows = list(graph.graph.query(comp_query))
        if debug:
            print(f"Composer-based candidates (composer = {tak_label}): {len(comp_rows)}")

        for row in comp_rows:
            uri = str(row["movie"])
            qid = extract_qid_from_uri(uri)
            if not qid:
                continue

            rating = None
            r_term = row.get("rating")
            if r_term is not None:
                try:
                    rating = float(r_term)
                except Exception:
                    rating = None

            if qid not in candidates_by_qid:
                candidates_by_qid[qid] = {
                    "qid": qid,
                    "uri": uri,
                    "label": qid2label.get(qid, None),
                    "rating": rating,
                    "sources": set(["composer"]),
                }
            else:
                item = candidates_by_qid[qid]
                item["sources"].add("composer")
                if item["rating"] is None and rating is not None:
                    item["rating"] = rating
    else:
        if debug:
            print("[Info] No Takemitsu QID; composer-based branch disabled.")

    # --- 两个属性都没拿到 → fallback ---
    if not candidates_by_qid:
        if debug:
            print("\n[Fallback] No attribute-based candidates; using hybrid graph+embedding instead.\n")
        return hybrid_graph_embedding_recommendation(
            seed_titles=[seed_title],
            max_graph_neighbors=200,
            and_threshold=1,
            top_k=top_k,
            debug=debug,
        )

    candidates = list(candidates_by_qid.values())

    # 排序：先源数（lang+composer > 单一），再 rating
    def sort_key(it):
        src_score = 0
        if "lang" in it["sources"]:
            src_score += 1
        if "composer" in it["sources"]:
            src_score += 1
        r = it["rating"]
        return (-src_score, 0 if r is None else -r)

    candidates.sort(key=sort_key)

    if debug:
        print(f"\nTotal combined candidates (language ∪ composer, excluding seed): {len(candidates)}")
        print(f"Top {top_k}:")
        for i, item in enumerate(candidates[:top_k], start=1):
            src = ",".join(sorted(item["sources"]))
            print(
                f"  #{i:2d} {item['label'] or item['qid']} "
                f"(qid={item['qid']}, rating={item['rating']}, sources={src})"
            )

    return candidates[:top_k]


# ---------- 再跑一遍 Japanese + Meryl 测试 ----------

print("\n======================================================================")
print("Re-running attribute-based recommendation tests (Japanese + Meryl)\n")

jp_recs = recommend_in_japanese_like_twin_sisters(top_k=15, debug=True)

print("\n----------------------------------------------------------------------")

meryl_recs = recommend_biographical_meryl_streep(top_k=15, debug=True)



Re-running attribute-based recommendation tests (Japanese + Meryl)

=== Attribute-based Japanese-language / Takemitsu rec ===
Question: What other movies in Japanese do you recommend? I liked Twin Sisters of Kyoto.
Seed: 'Twin Sisters of Kyoto' -> Q1458080 [Twin Sisters of Kyoto]
Language 'Japanese' -> Q5287 [Japanese]
Composer 'Toru/Tōru Takemitsu' -> Q155467 [Tōru Takemitsu]

Language-based candidates (original language = Japanese): 237
Composer-based candidates (composer = Tōru Takemitsu): 9

Total combined candidates (language ∪ composer, excluding seed): 238
Top 15:
  # 1 Dodes'ka-den (qid=Q1634355, rating=7.4, sources=composer,lang)
  # 2 Empire of Passion (qid=Q1659440, rating=7.0, sources=composer,lang)
  # 3 Rikyu (qid=Q7334152, rating=None, sources=composer,lang)
  # 4 Kwaidan (qid=Q133488, rating=None, sources=composer,lang)
  # 5 Harakiri (qid=Q1584406, rating=None, sources=composer,lang)
  # 6 The Woman in the Dunes (qid=Q1207936, rating=None, sources=composer,lang)
  # 7

# Final Version Of Recommend

In [52]:
# ============================================================
# Evaluation of current recommendation strategies
# ============================================================
#
# Strategies used here:
#   - Graph-based AND/OR over seeds:
#       graph_recommendation_and_or(seed_titles, top_k=...)
#       (director + genre neighbors, with simple AND/OR semantics and rating)
#
#   - Attribute-based:
#       recommend_in_japanese_like_twin_sisters(top_k=...)
#       recommend_biographical_meryl_streep(top_k=...)
#
# No calls to:
#   - hybrid_graph_embedding_recommendation
#   - embedding_recommendation_and_or_qid
#   - debug=..., and_mode=...
#
# This cell:
#   1. Runs the 6 official benchmark questions.
#   2. Compares our recommendations to the given example lists.
#   3. Runs a few simulated “similar-pattern” questions.

from typing import List, Dict, Any, Tuple

# --------- Safety checks ------------------------------------------------------

missing = []
if "graph_recommendation_and_or" not in globals():
    missing.append("graph_recommendation_and_or")
if "recommend_in_japanese_like_twin_sisters" not in globals():
    missing.append("recommend_in_japanese_like_twin_sisters")
if "recommend_biographical_meryl_streep" not in globals():
    missing.append("recommend_biographical_meryl_streep")

if missing:
    raise RuntimeError(
        "The following functions are missing; please run the cells that define them first:\n"
        + ", ".join(missing)
    )

# --------- Small utilities ----------------------------------------------------


def normalize_title(t: str) -> str:
    """Normalize a movie title for comparison."""
    return t.strip().lower()


def evaluate_against_gold(
    rec_labels: List[str], gold_titles: List[str]
) -> Tuple[List[str], List[str]]:
    """
    Compare recommended labels against example titles.
    Returns (overlap, missing) using original gold strings.
    """
    rec_norm = {normalize_title(l) for l in rec_labels}
    gold_norm_map = {normalize_title(g): g for g in gold_titles}

    overlap = []
    missing = []
    for g in gold_titles:
        if normalize_title(g) in rec_norm:
            overlap.append(g)
        else:
            missing.append(g)

    return overlap, missing


def print_eval_header(name: str, question: str):
    print("\n" + "=" * 70)
    print(f"TEST: {name}")
    print("-" * 70)
    print(question)
    print("-" * 70)


def print_eval_result(
    rec_labels: List[str], gold_titles: List[str], top_k_print: int = 10
):
    n_show = min(len(rec_labels), top_k_print)
    print(f"Top-{n_show} recommended titles:")
    for i, lbl in enumerate(rec_labels[:top_k_print], start=1):
        print(f"  #{i:2d} {lbl}")

    if gold_titles:
        overlap, missing = evaluate_against_gold(rec_labels, gold_titles)
        print(f"\nOverlap with gold examples: {len(overlap)}/{len(gold_titles)}")
        if overlap:
            print("  Hit examples:", overlap)
        if missing:
            print("  Gold not hit (for reference):", missing)


# --------- Benchmark questions and example answer sets ------------------------

BENCHMARKS: Dict[str, Dict[str, Any]] = {
    "musical_1": {
        "question": (
            "I like Singin' in the Rain, and Moulin Rouge. "
            "What other movies would you recommend for me to watch? "
            "Answers would include musical films from the 1950s."
        ),
        "seed_titles": ["Singin' in the Rain", "Moulin Rouge"],
        "gold": [
            "The Great Caruso",
            "On the Riviera",
            "Starlift",
            "The Medium",
            "The White-haired Girl",
            "Alice in Wonderland",
            "An American in Paris",
            "The Tales of Hoffmann",
            "Show Boat",
            "Limelight",
            "With a Song in My Heart",
            "Hans Christian Andersen",
            "April in Paris",
            "The Merry Widow",
            "Baiju Bawra",
            "The Jazz Singer",
            "The Band Wagon",
            "The 5,000 Fingers of Dr. T.",
            "On the Town Vol. 2",
            "When My Baby Smiles at Me Reloaded",
            "Peter Pan",
            "Torch Song",
            "Kiss Me Kate",
            "How to Marry a Millionaire",
            "Gentlemen Prefer Blondes",
            "Call Me Madam",
            "Lili",
            "Overture to The Merry Wives of Windsor",
            "Easter Parade Reloaded",
            "Calamity Jane",
        ],
        "mode": "graph",
    },
    "fellini": {
        "question": (
            "Recommend movies similar to La Dolce Vita and The Voice of the Moon. "
            "Adequate recommendations may include any movies from Italian director "
            "Federico Fellini or any Italian drama films."
        ),
        "seed_titles": ["La Dolce Vita", "The Voice of the Moon"],
        "gold": [
            "City of Women",
            "Satyricon",
            "Nights of Cabiria",
            "Intervista",
            "Ginger and Fred",
            "Amarcord",
            "Spirits of the Dead",
            "8½",
            "Fellini's Casanova Part 2",
            "Fellini's Casanova",
            "And the Ship Sails On",
            "Juliet of the Spirits",
            "La Strada",
            "Roma",
            "I Vitelloni",
            "Macaroni",
            "Le Bal",
            "We All Loved Each Other So Much",
            "Down and Dirty",
            "Unfair Competition",
            "I nuovi mostri",
            "A Special Day",
            "That Night in Varennes",
            "The Family",
        ],
        "mode": "graph",
    },
    "chicago_geisha_alice": {
        "question": (
            "I really enjoyed Chicago, Memoirs of a Geisha, and Alice in Wonderland. "
            "Can you recommend me some similar movies? Adequate recommendations "
            "include any movie where Colleen Atwood was the costume designer or any "
            "movie, winner or nominated, for the Academy Award for Best Costume Design."
        ),
        "seed_titles": ["Chicago", "Memoirs of a Geisha", "Alice in Wonderland"],
        "gold": [
            "Philadelphia",
            "Fantastic Beasts: The Crimes of Grindelwald",
            "Head Above Water",
            "Someone to Watch Over Me",
            "Mission: Impossible III",
            "Big Fish",
            "Sleepy Hollow",
            "The Silence of the Lambs",
            "Lemony Snicket's A Series of Unfortunate Events",
            "Nine",
            "Big Eyes",
            "Lorenzo's Oil",
            "Into the Woods",
            "Alice Through the Looking Glass",
            "Dark Shadows",
            "Little Women",
            "Mars Attacks!",
            "Manhunter",
            "Ed Wood",
            "Edward Scissorhands",
            "The Tourist",
            "Snow White and the Huntsman",
            "Fantastic Beasts and Where to Find Them",
            "The Tourist Vol. 2",
            "Miss Peregrine's Home for Peculiar Children",
            "Torch Song Trilogy",
            "Sweeney Todd: The Demon Barber of Fleet Street",
            "Public Enemies",
            "Chicago Reloaded",
            "Planet of the Apes",
            "Wyatt Earp",
            "Star Wars: Episode IV – A New Hope",
            "Gandhi",
            "Cruella",
            "Mad Max: Fury Road",
            "Chariots of Fire",
            "Marie Antoinette",
            "Barry Lyndon",
            "The Grand Budapest Hotel",
        ],
        "mode": "graph",
    },
    "forrest_lotr": {
        "question": (
            "Recommend some movies similar to Forrest Gump and "
            "The Lord of the Rings: The Fellowship of the Ring. "
            "Adequate recommendations include mostly the 15 top user-rated movies."
        ),
        "seed_titles": ["Forrest Gump", "The Lord of the Rings: The Fellowship of the Ring"],
        "gold": [
            "Acidulous Midtime Shed",
            "Yarest Aprepitant Grind",
            "Ardh Satya II",
            "Chicken-livered Jopadhola Let",
            "11'09\"01 September 11 2",
            "Rafter Romance Returns",
            "Leading Whin Slit",
            "Tranquilizing Oonopid Fly",
            "Death Spa Returns",
            "Fear of a Black Hat II",
            "Goliardic Judas cradle Do",
            "A Time to Kill II",
            "The Lord of the Rings: The Fellowship of the Ring",
            "Forrest Gump",
            "Rigorous Parsonage Throw",
        ],
        "mode": "graph",
    },
    "japanese_kyoto": {
        "question": (
            "What other movies in Japanese do you recommend? I liked Twin Sisters of Kyoto. "
            "Answers include any other movies with composer Tōru Takemitsu or any other movies in Japanese."
        ),
        "seed_titles": ["Twin Sisters of Kyoto"],
        "gold": [
            "Dodes'ka-den",
            "The Woman in the Dunes",
            "The Fossil",
            "Empire of Passion",
            "Sandakan No. 8",
            "The Burmese Harp",
            "Immortal Love",
            "Immortal Love Vol. 2",
            "Dragon Ball Super: Super Hero",
            "Antarctica",
            "Untamed",
            "Portrait of Chieko",
            "Late Autumn",
            "Fires on the Plain",
        ],
        "mode": "attr_japanese",
    },
    "meryl_bio": {
        "question": "Can you recommend some biographical movies given that I like Meryl Streep?",
        "seed_titles": [],
        "gold": [
            "Florence Foster Jenkins",
            "The Iron Lady",
            "Julie & Julia",
            "The Post",
            "Julia",
            "Out of Africa",
            "Music of the Heart",
            "Evil Angels",
            "Silkwood",
        ],
        "mode": "attr_meryl_bio",
    },
}

# --------- Run benchmark tests -----------------------------------------------

all_results: Dict[str, List[str]] = {}

print("\n" + "=" * 70)
print("Running benchmark tests for the 6 main questions")
print("=" * 70)

for key, info in BENCHMARKS.items():
    question = info["question"]
    mode = info["mode"]
    seeds = info["seed_titles"]
    gold = info["gold"]

    print_eval_header(key, question)

    if mode == "graph":
        # Graph-based multi-seed recommendation (director + genre + AND/OR + rating)
        recs = graph_recommendation_and_or(
            seeds,
            top_k=20,  # adjust if you want a different cut
        )
        labels = [r["label"] for r in recs]

    elif mode == "attr_japanese":
        # Attribute-based: Japanese language + Takemitsu
        recs = recommend_in_japanese_like_twin_sisters(top_k=20)
        labels = [r["label"] for r in recs]

    elif mode == "attr_meryl_bio":
        # Attribute-based: Meryl Streep ∧ biographical film
        recs = recommend_biographical_meryl_streep(top_k=20)
        labels = [r["label"] for r in recs]

    else:
        raise ValueError(f"Unknown mode: {mode}")

    all_results[key] = labels
    print_eval_result(labels, gold, top_k_print=10)

print("\n" + "=" * 70)
print("Finished benchmark tests for the 6 main questions.")
print("=" * 70)

# --------- Simulated similar-pattern tests -----------------------------------

SIMULATED_GROUPS = {
    "musical_2": {
        "description": "Musicals like 'An American in Paris' and 'The Band Wagon'",
        "seed_titles": ["An American in Paris", "The Band Wagon"],
        "mode": "graph",
    },
    "fellini_2": {
        "description": "Fellini-style: 'La Strada' and '8½'",
        "seed_titles": ["La Strada", "8½"],
        "mode": "graph",
    },
    "costume_2": {
        "description": "Colleen Atwood-like: 'Edward Scissorhands' and 'Sleepy Hollow'",
        "seed_titles": ["Edward Scissorhands", "Sleepy Hollow"],
        "mode": "graph",
    },
    "epic_2": {
        "description": (
            "Epic/fantasy: 'The Lord of the Rings: The Fellowship of the Ring' "
            "and 'The Lord of the Rings: The Return of the King'"
        ),
        "seed_titles": [
            "The Lord of the Rings: The Fellowship of the Ring",
            "The Lord of the Rings: The Return of the King",
        ],
        "mode": "graph",
    },
}

print("\n" + "=" * 70)
print("Running simulated similar-pattern tests (graph-based)")
print("=" * 70)

for key, info in SIMULATED_GROUPS.items():
    print("\n" + "-" * 70)
    print(f"SIMULATED TEST: {key}")
    print(info["description"])
    print("-" * 70)

    seeds = info["seed_titles"]
    mode = info["mode"]

    if mode == "graph":
        recs = graph_recommendation_and_or(
            seeds,
            top_k=15,
        )
        labels = [r["label"] for r in recs]
    else:
        raise ValueError(f"Unknown mode: {mode}")

    print("Seeds:", seeds)
    print("Top-10 simulated recommendations:")
    for i, lbl in enumerate(labels[:10], start=1):
        print(f"  #{i:2d} {lbl}")

print("\n" + "=" * 70)
print("All tests (benchmarks + simulated) completed.")
print("=" * 70)



Running benchmark tests for the 6 main questions

TEST: musical_1
----------------------------------------------------------------------
I like Singin' in the Rain, and Moulin Rouge. What other movies would you recommend for me to watch? Answers would include musical films from the 1950s.
----------------------------------------------------------------------

Graph AND/OR recommendation for seeds: ["Singin' in the Rain", 'Moulin Rouge']
  Seed "Singin' in the Rain" -> uri=http://www.wikidata.org/entity/Q309153 qid=Q309153 label=Singin' in the Rain movie=True
  Seed 'Moulin Rouge' -> uri=http://www.wikidata.org/entity/Q1508611 qid=Q1508611 label=Moulin Rouge movie=True

=== Graph neighbors for seed QID Q309153 (http://www.wikidata.org/entity/Q309153) ===
Total raw rows: 50
Sample neighbors (up to 5):
  Q1062362  label='Love Songs'  ratings=[None]
  Q685245  label='The Fabulous Baker Boys'  ratings=[None]
  Q2291842  label="Who's That Girl"  ratings=[4.8]
  Q7560207  label='Something fo

In [53]:
# ======================================================================
# Final dispatcher cell: unified entry point for all recommendation strategies
# ======================================================================

from typing import List, Dict, Any, Tuple

# ---- Strategy selection ------------------------------------------------

def _select_strategy(question: str, seeds: List[str]) -> str:
    """
    Decide which recommendation strategy to use based on the question text
    and the list of seed movie titles.

    Current logic:

    - If the question is about "Japanese" or explicitly mentions
      "Twin Sisters of Kyoto" / Takemitsu -> attribute-based JP/Takemitsu.
    - If the question mentions "Meryl Streep" and biographical movies ->
      attribute-based Meryl biopics.
    - Otherwise:
        * if we have >=2 seeds -> graph / hybrid multi-seed similarity
        * if we have 1 seed   -> graph / hybrid single-seed neighbors
    """
    q = question.lower()

    # Attribute-based: Japanese language / Tōru Takemitsu
    if (
        "japanese" in q
        or "twin sisters of kyoto".lower() in q
        or "toru takemitsu" in q
        or "tōru takemitsu" in q
    ):
        return "attr_japanese_takemitsu"

    # Attribute-based: Meryl Streep + biographical
    if "meryl streep" in q and ("biograph" in q or "biopic" in q or "bio " in q):
        return "attr_meryl_biographical"

    # Multi-seed: treat as “similar to these movies”
    if len(seeds) >= 2:
        return "multi_seed_graph_or_hybrid"

    # Single-seed fallback
    if len(seeds) == 1:
        return "single_seed_graph_or_hybrid"

    # No seeds & no attribute keyword: we don't know what to do
    return "unknown"


# ---- Main dispatcher ---------------------------------------------------

def recommend_movies(
    question: str,
    seeds: List[str],
    top_k: int = 20,
    debug: bool = True,
) -> Tuple[str, List[Dict[str, Any]]]:
    """
    High-level recommendation entry point used by the agent / notebook.

    Parameters
    ----------
    question : str
        Natural-language user question.
    seeds : List[str]
        Seed movie titles already extracted / linked by the upstream agent
        (e.g., ["Forrest Gump", "The Lord of the Rings: The Fellowship of the Ring"]).
    top_k : int
        Number of recommendations to return.
    debug : bool
        If True, underlying functions may print diagnostic information.

    Returns
    -------
    strategy_name : str
        Which strategy was chosen.
    recs : List[Dict[str, Any]]
        Recommendation objects, each typically containing at least:
        - "qid"
        - "label"
        - "avg_rating" or "rating"
        - plus any extra fields the underlying function adds.
    """
    strategy = _select_strategy(question, seeds)

    # 1) Attribute-based: Japanese language / Tōru Takemitsu
    if strategy == "attr_japanese_takemitsu":
        recs = recommend_in_japanese_like_twin_sisters(
            top_k=top_k,
            debug=debug,
        )
        return strategy, recs

    # 2) Attribute-based: Meryl Streep biographical films
    if strategy == "attr_meryl_biographical":
        recs = recommend_biographical_meryl_streep(
            top_k=top_k,
            debug=debug,
        )
        return strategy, recs

    # 3) Graph / Hybrid: multi-seed & single-seed cases
    #    Prefer hybrid if it is defined in the current notebook namespace,
    #    otherwise fall back to pure graph-based neighbors.
    use_hybrid = "hybrid_graph_embedding_recommendation" in globals()

    if strategy in ("multi_seed_graph_or_hybrid", "single_seed_graph_or_hybrid"):
        if use_hybrid:
            recs = hybrid_graph_embedding_recommendation(
                seeds,
                top_k=top_k,
                debug=debug,
            )
            return f"{strategy}__hybrid", recs
        else:
            recs = graph_recommendation_and_or(
                seeds,
                top_k=top_k,
                debug=debug,
            )
            return f"{strategy}__graph", recs

    # 4) Fallbacks when strategy==unknown:
    #    try attribute-based hints first, then graph.

    q = question.lower()
    if "japanese" in q and "recommend_in_japanese_like_twin_sisters" in globals():
        recs = recommend_in_japanese_like_twin_sisters(
            top_k=top_k,
            debug=debug,
        )
        return "fallback_attr_japanese", recs

    if "meryl streep" in q and "recommend_biographical_meryl_streep" in globals():
        recs = recommend_biographical_meryl_streep(
            top_k=top_k,
            debug=debug,
        )
        return "fallback_attr_meryl", recs

    if seeds:
        # Last-resort graph neighbors
        recs = graph_recommendation_and_or(
            seeds,
            top_k=top_k,
            debug=debug,
        )
        return "fallback_graph_neighbors", recs

    raise RuntimeError(
        "recommend_movies: could not determine a suitable strategy "
        "and no seeds were provided."
    )


# ---- Helper to pretty-print results ------------------------------------

def pretty_print_recommendations(
    question: str,
    seeds: List[str],
    top_k: int = 10,
    debug: bool = False,
) -> List[Dict[str, Any]]:
    """
    Convenience wrapper: run recommend_movies() and print a compact summary,
    similar to the benchmark output you already used.
    """
    strategy, recs = recommend_movies(
        question=question,
        seeds=seeds,
        top_k=top_k,
        debug=debug,
    )

    print("======================================================================")
    print("Question:")
    print(question)
    print("----------------------------------------------------------------------")
    print(f"Strategy used: {strategy}")
    print(f"Seeds: {seeds}")
    print("Top recommendations:")

    for i, info in enumerate(recs[:top_k], start=1):
        label = info.get("label") or info.get("title") or "<unknown>"
        rating = info.get("avg_rating", info.get("rating", None))
        if rating is not None:
            print(f"  # {i} {label}  (rating={rating})")
        else:
            print(f"  # {i} {label}")

    return recs


# ****FULL Strategy Recommendation***

In [54]:
import numpy as np

# --------------------------------------------------------------------
# SPARQL helper
# --------------------------------------------------------------------

# Prefixes for our local KG (same as used elsewhere in the project)
PREFIXES = """
PREFIX ddis: <http://ddis.ch/atai/>
PREFIX wd:   <http://www.wikidata.org/entity/>
PREFIX wdt:  <http://www.wikidata.org/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
"""

def run_sparql(query: str):
    """
    Execute a SPARQL SELECT on the local rdflib graph and return list of dicts.
    Assumes a global `graph` object with `.graph` (rdflib.Graph).
    """
    res = graph.graph.query(query)
    out = []
    vars_ = [str(v) for v in res.vars]
    for row in res:
        d = {}
        for v in vars_:
            val = getattr(row, v, None)
            if val is not None:
                d[v] = str(val)
        out.append(d)
    return out


# Simple helpers on QIDs / labels
def get_label(qid: str) -> str:
    return qid2label.get(qid, qid)


# --------------------------------------------------------------------
# 1) Hybrid graph + embedding recommender (你的最新版)
# --------------------------------------------------------------------

def graph_embedding_hybrid_recommendation(
    seed_titles,
    k_graph_per_seed: int = 200,
    top_k: int = 10,
    use_and: bool = True,
    print_debug: bool = True,
):
    """
    Hybrid recommender:
      1) Use graph neighbors (same director / same genre) as candidate set.
      2) Rank primarily by graph structure (coverage, rating),
         and only secondarily by embeddings.

    Return format: list of (qid, info_dict)
      info_dict has at least: label, coverage, avg_rating, emb_dist, has_emb.
    """

    print("\n" + "=" * 70)
    print("Hybrid graph+embedding recommendation for seeds:", seed_titles)

    # 1. Link titles to movie URIs/QIDs
    seeds = []
    for title in seed_titles:
        uri, lbl, is_movie = link_movie(title, must_be_movie=True)
        if uri is None or not is_movie:
            if print_debug:
                print(f"  Seed {title!r}: could not link to a movie (uri={uri}, is_movie={is_movie}). Skipping.")
            continue
        qid = uri.rsplit("/", 1)[-1]
        seeds.append({"title": title, "uri": uri, "qid": qid, "label": lbl})

    if not seeds:
        print("❌ No seeds could be linked to movies; aborting.")
        return []

    if print_debug:
        print("\nLinked seeds:")
        for s in seeds:
            print(f"  {s['title']!r} -> uri={s['uri']} qid={s['qid']} label={s['label']}")

    # 2. Graph neighbors per seed
    neighbor_sets = []
    candidates = {}  # qid -> info dict

    for s in seeds:
        uri = s["uri"]
        qid_seed = s["qid"]

        query = composer.compose_graph_rec_query(uri)
        if k_graph_per_seed != 50:
            # crude, but matches earlier logic
            query = query.replace("LIMIT 50", f"LIMIT {k_graph_per_seed}")

        rows = run_sparql(query)
        qids_local = set()

        if print_debug:
            print(f"\n=== Graph neighbors for seed QID {qid_seed} ({uri}) ===")
            print(f"Total raw rows: {len(rows)}")

        for i, row in enumerate(rows):
            movie_uri = row.get("movie")
            if movie_uri is None:
                continue
            cqid = movie_uri.rsplit("/", 1)[-1]
            qids_local.add(cqid)

            rating_str = row.get("rating")
            rating = None
            if rating_str is not None:
                try:
                    rating = float(rating_str)
                except ValueError:
                    rating = None

            if cqid not in candidates:
                candidates[cqid] = {
                    "label": get_label(cqid),
                    "ratings": [],
                    "coverage": 0,
                    "from_seeds": set(),
                }
            candidates[cqid]["coverage"] += 1
            candidates[cqid]["from_seeds"].add(qid_seed)
            if rating is not None:
                candidates[cqid]["ratings"].append(rating)

        neighbor_sets.append(qids_local)

        if print_debug and rows:
            print("Sample neighbors (up to 5):")
            for row in rows[:5]:
                cqid = row["movie"].rsplit("/", 1)[-1]
                lab = get_label(cqid)
                rating_str = row.get("rating")
                print(f"  {cqid}  label='{lab}'  ratings={[rating_str] if rating_str else [None]}")

    # Remove seeds from candidates and neighbor sets
    seed_qids = {s["qid"] for s in seeds}
    for sq in seed_qids:
        candidates.pop(sq, None)
    for sset in neighbor_sets:
        sset.difference_update(seed_qids)

    # 3. Decide active candidate set: AND vs OR
    and_qids = set.intersection(*neighbor_sets) if (use_and and neighbor_sets) else set()
    if print_debug:
        sizes = [len(s) for s in neighbor_sets]
        print("\n  Per-seed neighbor sizes:", sizes)
        print("  AND intersection size:", len(and_qids))

    if use_and and len(and_qids) > 0:
        active_qids = and_qids
        if print_debug:
            print("  Using AND semantics (common neighbors across all seeds).")
    else:
        active_qids = set().union(*neighbor_sets)
        if print_debug:
            print("  AND set empty or disabled, using OR-union.")

    active_candidates = {qid: candidates[qid] for qid in active_qids if qid in candidates}
    if not active_candidates:
        print("❌ No graph-based candidates found; aborting.")
        return []

    # 4. Prepare seed embeddings
    seed_emb_indices = [qid2idx[s["qid"]] for s in seeds if s["qid"] in qid2idx]
    if not seed_emb_indices:
        if print_debug:
            print("⚠️ No seed has an embedding; falling back to pure graph ranking.")
        for info in active_candidates.values():
            if info["ratings"]:
                info["avg_rating"] = float(sum(info["ratings"]) / len(info["ratings"]))
            else:
                info["avg_rating"] = 0.0
            info["emb_dist"] = None
            info["has_emb"] = False
    else:
        seed_emb = E[seed_emb_indices]
        for qid, info in active_candidates.items():
            # rating
            if info["ratings"]:
                info["avg_rating"] = float(sum(info["ratings"]) / len(info["ratings"]))
            else:
                info["avg_rating"] = 0.0

            # embedding dist
            if qid in qid2idx:
                idx = qid2idx[qid]
                v = E[idx]
                diffs = seed_emb - v
                dists = np.sum(diffs * diffs, axis=1)
                emb_dist = float(dists.mean())
                info["emb_dist"] = emb_dist
                info["has_emb"] = True
            else:
                info["emb_dist"] = None
                info["has_emb"] = False

    # 5. Sort: embedding only as tie-breaker, graph first
    def sort_key(item):
        qid, info = item
        has_emb = info.get("has_emb", False)
        emb_dist = info["emb_dist"] if info["emb_dist"] is not None else 1e9
        coverage = info["coverage"]
        avg_rating = info["avg_rating"]
        label = info["label"]

        return (
            0 if has_emb else 1,  # prefer items with embeddings
            -coverage,            # more seeds in common first
            -avg_rating,          # higher rating first
            emb_dist,             # closer embeddings first
            label,
        )

    ranked_items = sorted(active_candidates.items(), key=sort_key)[:top_k]

    # 6. Pretty print
    print("\n  Final ranked hybrid recommendations:")
    for qid, info in ranked_items:
        emb_str = f"{info['emb_dist']:.4f}" if info["emb_dist"] is not None else "None"
        print(
            f"    -> {info['label']}  "
            f"(qid={qid}, emb_dist={emb_str}, "
            f"coverage={info['coverage']}, avg_rating={info['avg_rating']:.2f})"
        )

    top_labels = [info["label"] for qid, info in ranked_items]
    print("Top labels:", top_labels)
    return ranked_items


# --------------------------------------------------------------------
# 2) Attribute-based recommenders (Japanese/Takemitsu + Meryl/biopic)
# --------------------------------------------------------------------

# Hard-coded important QIDs (confirmed from earlier debug prints)
JAPANESE_LANG_QID      = "Q5287"    # Japanese
TAKEMITSU_QID          = "Q155467"  # Tōru Takemitsu
MERYL_STREEP_QID       = "Q873"     # Meryl Streep
BIOGRAPHICAL_FILM_QID  = "Q645928"  # biographical film


def recommend_japanese_takemitsu_like(
    seed_title: str,
    top_k: int = 15,
    debug: bool = True,
):
    """
    Attribute-based recommender for:

      "What other movies in Japanese do you recommend? I liked Twin Sisters of Kyoto.
       Answers include any other movies with composer Tōru Takemitsu or any other
       movies in Japanese"

    Strategy:
      - Candidates = movies whose original language is Japanese (P364 = Q5287)
                     ∪ movies whose composer is Takemitsu (P86 = Q155467)
      - Exclude the seed movie
      - Sort by:
          1) number of sources (both -> 2, only one -> 1)
          2) rating (ddis:rating)
    Return: list of (qid, info_dict)
    """

    print("\n=== Attribute-based Japanese-language / Takemitsu rec ===")
    print(f"Question: What other movies in Japanese do you recommend? I liked {seed_title}.")
    # Resolve seed
    seed_uri, seed_label, is_movie = link_movie(seed_title, must_be_movie=True)
    seed_qid = None
    if seed_uri is not None and is_movie:
        seed_qid = seed_uri.rsplit("/", 1)[-1]
        print(f"Seed: {seed_title!r} -> {seed_qid} [{seed_label}]")
    else:
        print(f"⚠️ Could not resolve seed {seed_title!r} as a movie.")

    print(f"Language 'Japanese' -> {JAPANESE_LANG_QID} [{get_label(JAPANESE_LANG_QID)}]")
    print(f"Composer 'Toru/Tōru Takemitsu' -> {TAKEMITSU_QID} [{get_label(TAKEMITSU_QID)}]")

    # 1) Japanese-language candidates
    lang_query = PREFIXES + """
    SELECT DISTINCT ?movie ?rating WHERE {
      ?movie wdt:P31 wd:Q11424 .               # instance of film
      ?movie wdt:P364 wd:Q5287 .              # original language = Japanese
      OPTIONAL { ?movie ddis:rating ?rating . }
    }
    """
    lang_rows = run_sparql(lang_query)

    # 2) Takemitsu composer candidates
    tak_query = PREFIXES + """
    SELECT DISTINCT ?movie ?rating WHERE {
      ?movie wdt:P31 wd:Q11424 .
      ?movie wdt:P86 wd:Q155467 .             # composer = Tōru Takemitsu
      OPTIONAL { ?movie ddis:rating ?rating . }
    }
    """
    tak_rows = run_sparql(tak_query)

    if debug:
        print(f"\nLanguage-based candidates (original language = Japanese): {len(lang_rows)}")
        print(f"Composer-based candidates (composer = Tōru Takemitsu): {len(tak_rows)}")

    # Combine
    candidates = {}  # qid -> info
    def add_row(row, source_tag):
        uri = row.get("movie")
        if not uri:
            return
        qid = uri.rsplit("/", 1)[-1]
        if seed_qid and qid == seed_qid:
            return
        rating_str = row.get("rating")
        rating = None
        if rating_str is not None:
            try:
                rating = float(rating_str)
            except ValueError:
                rating = None
        if qid not in candidates:
            candidates[qid] = {
                "label": get_label(qid),
                "ratings": [],
                "sources": set(),
            }
        if rating is not None:
            candidates[qid]["ratings"].append(rating)
        candidates[qid]["sources"].add(source_tag)

    for r in lang_rows:
        add_row(r, "lang")
    for r in tak_rows:
        add_row(r, "composer")

    if not candidates:
        print("\n[Fallback] No attribute-based candidates; using hybrid graph+embedding instead.")
        if seed_title:
            return graph_embedding_hybrid_recommendation(
                [seed_title],
                k_graph_per_seed=200,
                top_k=top_k,
                use_and=True,
                print_debug=debug,
            )
        return []

    # Compute avg_rating
    for qid, info in candidates.items():
        if info["ratings"]:
            info["avg_rating"] = float(sum(info["ratings"]) / len(info["ratings"]))
        else:
            info["avg_rating"] = 0.0

    # Sort: by number of sources (2 > 1), rating desc, label
    def sort_key(item):
        qid, info = item
        return (
            -len(info["sources"]),
            -info["avg_rating"],
            info["label"],
        )

    ranked_items = sorted(candidates.items(), key=sort_key)[:top_k]

    if debug:
        print(f"\nTotal combined candidates (language ∪ composer, excluding seed): {len(candidates)}")
        print(f"Top {top_k}:")
        for i, (qid, info) in enumerate(ranked_items, start=1):
            srcs = ",".join(sorted(info["sources"]))
            print(
                f"  # {i:2d} {info['label']} (qid={qid}, rating={info['avg_rating']:.1f}, sources={srcs})"
            )

    return ranked_items


def recommend_biographical_meryl_streep(
    top_k: int = 15,
    debug: bool = True,
):
    """
    Attribute-based recommender for:

      "Can you recommend some biographical movies given that I like Meryl Streep?"

    Strategy:
      - Candidates = movies that have:
          * cast member Meryl Streep (P161 = Q873) AND
          * genre biographical film (P136 = Q645928)
      - Sort by rating (ddis:rating).
    Return: list of (qid, info_dict)
    """

    print("\n=== Attribute-based biographical Meryl Streep rec ===")
    print("Question: Can you recommend some biographical movies given that I like Meryl Streep?")
    print(f"Meryl Streep -> {MERYL_STREEP_QID} [{get_label(MERYL_STREEP_QID)}]")
    print(f"Biographical genre -> {BIOGRAPHICAL_FILM_QID} [{get_label(BIOGRAPHICAL_FILM_QID)}]")

    query = PREFIXES + """
    SELECT DISTINCT ?movie ?rating WHERE {
      ?movie wdt:P31 wd:Q11424 .               # film
      ?movie wdt:P161 wd:Q873 .               # cast member Meryl Streep
      ?movie wdt:P136 wd:Q645928 .           # genre biographical film
      OPTIONAL { ?movie ddis:rating ?rating . }
    }
    """
    rows = run_sparql(query)

    candidates = {}
    for row in rows:
        uri = row.get("movie")
        if not uri:
            continue
        qid = uri.rsplit("/", 1)[-1]
        rating_str = row.get("rating")
        rating = None
        if rating_str is not None:
            try:
                rating = float(rating_str)
            except ValueError:
                rating = None
        if qid not in candidates:
            candidates[qid] = {
                "label": get_label(qid),
                "ratings": [],
            }
        if rating is not None:
            candidates[qid]["ratings"].append(rating)

    # Compute avg_rating
    for qid, info in candidates.items():
        if info["ratings"]:
            info["avg_rating"] = float(sum(info["ratings"]) / len(info["ratings"]))
        else:
            info["avg_rating"] = 0.0

    # Sort by rating desc, label
    ranked_items = sorted(
        candidates.items(),
        key=lambda item: (-item[1]["avg_rating"], item[1]["label"]),
    )[:top_k]

    if debug:
        print(f"\nIntersection (Meryl Streep & biographical): {len(candidates)} films")
        print(f"Total candidate films: {len(candidates)}")
        print(f"Top {top_k}:")
        for i, (qid, info) in enumerate(ranked_items, start=1):
            print(f"  # {i:2d} {info['label']} (qid={qid}, rating={info['avg_rating'] if info['avg_rating'] else 'None'})")

    return ranked_items


# --------------------------------------------------------------------
# 3) High-level router: decide which strategy to use per question key
# --------------------------------------------------------------------

def recommend_movies_for_question(
    question_key: str,
    seeds,
    question_text: str | None = None,
    top_k: int = 10,
    debug: bool = True,
):
    """
    Unified entry point used by the benchmark/test cell.

    - japanese_kyoto → attribute-based (Japanese language ∪ Takemitsu)
    - meryl_bio      → attribute-based (Meryl Streep ∧ biographical film)
    - others         → hybrid graph+embedding (multi-seed AND/OR)
    """
    if question_key == "japanese_kyoto":
        # seeds assumed to contain exactly one title, the liked movie
        seed_title = seeds[0] if seeds else "Twin Sisters of Kyoto"
        return recommend_japanese_takemitsu_like(
            seed_title=seed_title,
            top_k=top_k,
            debug=debug,
        )

    if question_key == "meryl_bio":
        return recommend_biographical_meryl_streep(
            top_k=top_k,
            debug=debug,
        )

    # Default: hybrid graph+embedding multi-seed
    return graph_embedding_hybrid_recommendation(
        seed_titles=seeds,
        k_graph_per_seed=200,
        top_k=top_k,
        use_and=True,
        print_debug=debug,
    )


In [55]:
# ==============================================================
# Benchmark tests for the 6 main questions
# ==============================================================

benchmark_questions = {
    "musical_1": {
        "question": (
            "I like Singin' in the Rain, and Moulin Rouge. "
            "What other movies would you recommend for me to watch? "
            "Answers would include musical films from the 1950s."
        ),
        "seeds": ["Singin' in the Rain", "Moulin Rouge"],
        "top_k": 10,
    },
    "fellini": {
        "question": (
            "Recommend movies similar to La Dolce Vita and The Voice of the Moon. "
            "The adequate recommendations may include any movies from Italian director "
            "Federico Fellini or any Italian drama films."
        ),
        "seeds": ["La Dolce Vita", "The Voice of the Moon"],
        "top_k": 10,
    },
    "chicago_geisha_alice": {
        "question": (
            "I really enjoyed Chicago, Memoirs of a Geisha, and Alice in Wonderland. "
            "Can you recommend me some similar movies? "
            "Adequate recommendations include any movie where Colleen Atwood was the "
            "costume designer or any movie, winner or nominated, for the Academy Award "
            "for Best Costume Design."
        ),
        "seeds": ["Chicago", "Memoirs of a Geisha", "Alice in Wonderland"],
        "top_k": 10,
    },
    "forrest_lotr": {
        "question": (
            "Recommend some movies similar to Forrest Gump and "
            "The Lord of the Rings: The Fellowship of the Ring. "
            "The adequate recommendations include mostly the 15 top user-rated movies."
        ),
        "seeds": ["Forrest Gump", "The Lord of the Rings: The Fellowship of the Ring"],
        "top_k": 10,
    },
    "japanese_kyoto": {
        "question": (
            "What other movies in Japanese do you recommend? I liked Twin Sisters of Kyoto. "
            "Answers include any other movies with composer Tōru Takemitsu or any other "
            "movies in Japanese."
        ),
        "seeds": ["Twin Sisters of Kyoto"],
        "top_k": 15,
    },
    "meryl_bio": {
        "question": (
            "Can you recommend some biographical movies given that I like Meryl Streep?"
        ),
        "seeds": [],  # seeds not needed for this one
        "top_k": 15,
    },
}

print("\n" + "=" * 70)
print("Running benchmark tests for the 6 main questions")
print("=" * 70 + "\n")

for key, cfg in benchmark_questions.items():
    question = cfg["question"]
    seeds    = cfg["seeds"]
    top_k    = cfg["top_k"]

    print("\n" + "=" * 70)
    print(f"TEST: {key}")
    print("-" * 70)
    print(question)
    print("-" * 70)

    results = recommend_movies_for_question(
        question_key=key,
        seeds=seeds,
        question_text=question,
        top_k=top_k,
        debug=True,   # show internal debug info
    )

    # Normalize to labels list for a compact summary
    labels = [info["label"] for (_, info) in results]
    print("\nTop titles (summary):")
    for rank, title in enumerate(labels, start=1):
        print(f"  # {rank:2d} {title}")

print("\n" + "=" * 70)
print("Finished benchmark tests for the 6 main questions.")
print("=" * 70 + "\n")


# ==============================================================
# Simulated similar-pattern tests (graph-based / hybrid)
# ==============================================================

simulated_tests = {
    "musical_2": {
        "question": "Musicals like 'An American in Paris' and 'The Band Wagon'",
        "seeds": ["An American in Paris", "The Band Wagon"],
        "top_k": 15,
    },
    "fellini_2": {
        "question": "Fellini-style: 'La Strada' and '8½'",
        "seeds": ["La Strada", "8½"],
        "top_k": 15,
    },
    "costume_2": {
        "question": "Colleen Atwood-like: 'Edward Scissorhands' and 'Sleepy Hollow'",
        "seeds": ["Edward Scissorhands", "Sleepy Hollow"],
        "top_k": 15,
    },
    "epic_2": {
        "question": (
            "Epic/fantasy: 'The Lord of the Rings: The Fellowship of the Ring' "
            "and 'The Lord of the Rings: The Return of the King'"
        ),
        "seeds": [
            "The Lord of the Rings: The Fellowship of the Ring",
            "The Lord of the Rings: The Return of the King",
        ],
        "top_k": 15,
    },
}

print("\n" + "=" * 70)
print("Running simulated similar-pattern tests (graph-based / hybrid)")
print("=" * 70 + "\n")

for key, cfg in simulated_tests.items():
    question = cfg["question"]
    seeds    = cfg["seeds"]
    top_k    = cfg["top_k"]

    print("\n" + "-" * 70)
    print(f"SIMULATED TEST: {key}")
    print(question)
    print("-" * 70)

    results = recommend_movies_for_question(
        question_key=key,   # falls back to hybrid for unknown keys
        seeds=seeds,
        question_text=question,
        top_k=top_k,
        debug=True,
    )

    labels = [info["label"] for (_, info) in results]
    print(f"Seeds: {seeds}")
    print("Top simulated recommendations:")
    for rank, title in enumerate(labels, start=1):
        print(f"  # {rank:2d} {title}")

print("\n" + "=" * 70)
print("All tests (benchmarks + simulated) completed.")
print("=" * 70)



Running benchmark tests for the 6 main questions


TEST: musical_1
----------------------------------------------------------------------
I like Singin' in the Rain, and Moulin Rouge. What other movies would you recommend for me to watch? Answers would include musical films from the 1950s.
----------------------------------------------------------------------

Hybrid graph+embedding recommendation for seeds: ["Singin' in the Rain", 'Moulin Rouge']

Linked seeds:
  "Singin' in the Rain" -> uri=http://www.wikidata.org/entity/Q309153 qid=Q309153 label=Singin' in the Rain
  'Moulin Rouge' -> uri=http://www.wikidata.org/entity/Q1508611 qid=Q1508611 label=Moulin Rouge

=== Graph neighbors for seed QID Q309153 (http://www.wikidata.org/entity/Q309153) ===
Total raw rows: 200
Sample neighbors (up to 5):
  Q1062362  label='Love Songs'  ratings=[None]
  Q685245  label='The Fabulous Baker Boys'  ratings=[None]
  Q2291842  label='Who's That Girl'  ratings=['4.8']
  Q7560207  label='Something for t